# Experiment: DAPO-Math-17K Sign-Flipped FEPO/Dr.GRPO Training with AMC23 Evaluation

This notebook runs FEPO training with supervised ground-truth rewards and Dr.GRPO-style group advantages.

Training uses the full `zhuzilin/dapo-math-17k` (`default/train`) split. Each DAPO row provides a chat-style `prompt` and `label`; the math question is extracted from the prompt, rebuilt through this notebook's existing chat prompt path, and rewarded against the normalized label. Solver and failure advantages are group-centered without standard-deviation normalization.

Evaluation uses `math-ai/amc23` (`test`) only. It evaluates batches of prompts and generates `CONFIG.eval_generations` answers per prompt, reporting `pass@1`, `pass@k`, and `avg@k`.

Runtime layout:
- GPU 0: vLLM 0.6.5 OpenAI-compatible HTTP server with dynamic LoRA loading.
- GPU 1: solver/failure LoRA training, frozen reference model for optional KL anchoring, clipped FEPO losses, logging, checkpointing, and AMC23 evaluation.


In [ ]:
!uv pip install wandb

import os
import sys
import site
import json
import shutil
import zipfile
import hashlib
import subprocess
import importlib.metadata as importlib_metadata
from pathlib import Path

CUSTOM_SITE = Path("/kaggle/working/custom_site")
INPUT_ROOT = Path("/kaggle/input/datasets/ismamswapnil/my-wheelhouse")
UNZIP_DIR = Path("/kaggle/working/wheelhouse_unzipped")
UV_PREFIX = Path("/kaggle/working/uv_prefix")
os.environ["UV_LINK_MODE"] = "copy"

CUSTOM_SITE.mkdir(parents=True, exist_ok=True)
UNZIP_DIR.mkdir(parents=True, exist_ok=True)
UV_PREFIX.mkdir(parents=True, exist_ok=True)

REQUIRED_PACKAGES = [
    "vllm==0.6.5",
    "numpy==1.26.4",
    "pandas==2.2.2",
    "transformers==4.47.1",
    "datasets==3.2.0",
    "peft==0.14.0",
    "accelerate==1.2.1",
    "bitsandbytes==0.45.0",
    # "wandb==0.19.1",
]

def add_custom_site_first(custom_site: Path):
    custom_site = str(custom_site)
    site.addsitedir(custom_site)
    if custom_site in sys.path:
        sys.path.remove(custom_site)
    sys.path.insert(0, custom_site)
    os.environ["PYTHONPATH"] = custom_site + os.pathsep + os.environ.get("PYTHONPATH", "")

def package_name(req: str):
    return req.split("==")[0].replace("_", "-").lower()

def package_version(req: str):
    return req.split("==")[1]

def installed_version(name: str):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None

def requirements_hash(requirements):
    text = "\n".join(sorted(requirements))
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]

def find_wheel_dir():
    wheels = list(INPUT_ROOT.rglob("*.whl"))

    if wheels:
        print(f"Found {len(wheels)} wheel files under /kaggle/input.")
        parent_counts = {}
        for w in wheels:
            parent_counts[w.parent] = parent_counts.get(w.parent, 0) + 1
        return max(parent_counts, key=parent_counts.get)

    zips = list(INPUT_ROOT.rglob("*.zip"))
    if not zips:
        raise FileNotFoundError(
            "No .whl or .zip files found under /kaggle/input. "
            "Attach the wheelhouse dataset to this notebook first, or enable internet and remove --offline."
        )

    print("No .whl found directly. Found zip files:")
    for z in zips[:20]:
        print(" -", z)

    for z in zips:
        print(f"Extracting {z} -> {UNZIP_DIR}")
        with zipfile.ZipFile(z, "r") as zip_ref:
            zip_ref.extractall(UNZIP_DIR)

    wheels = list(UNZIP_DIR.rglob("*.whl"))
    if not wheels:
        raise FileNotFoundError("Zip extracted, but no .whl files were found inside.")

    parent_counts = {}
    for w in wheels:
        parent_counts[w.parent] = parent_counts.get(w.parent, 0) + 1
    return max(parent_counts, key=parent_counts.get)

def ensure_uv_available(wheel_dir: Path):
    uv_path = shutil.which("uv")
    if uv_path:
        print("Found uv:", uv_path)
        return uv_path

    print("uv not found. Trying to install uv first...")

    os.environ["PATH"] = str(UV_PREFIX / "bin") + os.pathsep + os.environ.get("PATH", "")

    uv_wheels = list(wheel_dir.glob("uv-*.whl"))
    if uv_wheels:
        cmd = [
            sys.executable, "-m", "pip", "install",
            "--no-index",
            "--find-links", str(wheel_dir),
            "--prefix", str(UV_PREFIX),
            "uv",
        ]
        print("Installing uv from wheelhouse:")
    else:
        cmd = [
            sys.executable, "-m", "pip", "install",
            "--prefix", str(UV_PREFIX),
            "uv",
        ]
        print("No uv wheel found. Trying to install uv from PyPI. This requires internet:")

    print(" ".join(cmd))
    subprocess.check_call(cmd)

    uv_path = shutil.which("uv")
    if not uv_path:
        raise RuntimeError(
            "uv was installed but the uv executable was not found on PATH. "
            "Add a uv-*.whl to your wheelhouse or enable internet for this setup cell."
        )

    print("Installed uv:", uv_path)
    return uv_path

WHEEL_DIR = find_wheel_dir()

print("\nUsing wheel directory:", WHEEL_DIR)
print("\nFirst 20 wheels:")
for p in sorted(WHEEL_DIR.glob("*.whl"))[:20]:
    print(" -", p.name)

UV_BIN = ensure_uv_available(WHEEL_DIR)

add_custom_site_first(CUSTOM_SITE)

marker = CUSTOM_SITE / f".installed_uv_{requirements_hash(REQUIRED_PACKAGES)}"

need_install = not marker.exists()

for req in REQUIRED_PACKAGES:
    name = package_name(req)
    expected = package_version(req)
    found = installed_version(name)
    if found != expected:
        print(f"[version check] {name}: found={found}, expected={expected}. Will install.")
        need_install = True

if need_install:
    cmd = [
        UV_BIN, "pip", "install",
        "--offline",
        "--no-index",
        "--find-links", str(WHEEL_DIR),
        "--target", str(CUSTOM_SITE),
        "--python", sys.executable,
        "--reinstall",
        "--only-binary", ":all:",
        *REQUIRED_PACKAGES,
    ]

    print("\nInstalling exact packages with uv into:", CUSTOM_SITE)
    print(" ".join(cmd))
    subprocess.check_call(cmd)

    marker.write_text(json.dumps({
        "installer": "uv",
        "requirements": REQUIRED_PACKAGES,
        "custom_site": str(CUSTOM_SITE),
        "wheel_dir": str(WHEEL_DIR),
        "uv_bin": str(UV_BIN),
    }, indent=2))

    add_custom_site_first(CUSTOM_SITE)

print("\nVerifying versions:")

bad = []
for req in REQUIRED_PACKAGES:
    name = package_name(req)
    expected = package_version(req)
    found = installed_version(name)
    status = "OK" if found == expected else "MISMATCH"
    print(f"{status:8s} {name:15s} found={found} expected={expected}")
    if found != expected:
        bad.append((name, found, expected))

if bad:
    raise RuntimeError(f"Version mismatch after uv install: {bad}")

print("\nAll requested package versions are active.")
print("Custom site:", CUSTOM_SITE)
print("Wheel dir:", WHEEL_DIR)

# Must be set before torch is imported by vLLM/transformers for best effect.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Import smoke test. Keep this in the first setup cell to fail early if any wheel is missing/incompatible.
import numpy as np
import pandas as pd
import transformers
import datasets
import peft
import accelerate
import bitsandbytes
import vllm
import wandb

print("numpy:", np.__version__, np.__file__)
print("pandas:", pd.__version__, pd.__file__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("vllm:", vllm.__version__)
print("wandb:", wandb.__version__)


## 1. Install pinned Kaggle dependencies

Run this cell once at the start of a fresh Kaggle session. If pip replaces `torch`, `torchvision`, `numpy`, or CUDA-linked packages, restart the Kaggle session once after the install, then continue from the import/config cells.

Pinned versions:
- `vllm==0.6.5`
- `numpy==1.26.4`
- `pandas==2.2.2`
- `transformers==4.47.1`
- `datasets==3.2.0`
- `peft==0.14.0`
- `accelerate==1.2.1`
- `bitsandbytes==0.45.0`
- `wandb`


## 2. Imports, config, and environment checks

The active algorithm is FEPO with Dr.GRPO-style ground-truth answer advantages. Training data comes from the full DAPO-Math-17K training split, and evaluation data comes from AMC23.


In [ ]:
import gc
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass
from decimal import Decimal, InvalidOperation, getcontext
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    import wandb
except Exception:
    wandb = None

try:
    import bitsandbytes as bnb
except Exception as exc:
    raise RuntimeError("bitsandbytes is required for AdamW8bit. Re-run the install cell first.") from exc

getcontext().prec = 50


@dataclass
class GRPOConfig:
    model_id: str = "Qwen/Qwen2.5-1.5B-Instruct"
    train_dataset_name: str = "zhuzilin/dapo-math-17k"
    train_dataset_config: Optional[str] = "default"
    train_split: str = "train"
    train_example_limit: Optional[int] = None
    eval_dataset_name: str = "math-ai/amc23"
    eval_dataset_config: Optional[str] = None
    eval_split: str = "test"
    seed: int = 42

    # Kaggle two-T4 layout.
    vllm_visible_device: str = "0"
    train_device: str = "cuda:1"
    vllm_host: str = "127.0.0.1"
    vllm_port: int = 8000
    vllm_base_model_name: str = "qwen-base"
    vllm_lora_name: str = "train_lora"

    # vLLM/runtime memory controls.
    max_model_len: int = 4096
    max_prompt_tokens: int = 2048
    train_max_completion_tokens: int = 2048
    eval_max_completion_tokens: int = 2048
    max_num_seqs: int = 16
    vllm_gpu_memory_utilization: float = 0.90
    vllm_enforce_eager: bool = False
    vllm_allow_eager_fallback: bool = True
    vllm_attention_backend: str = "XFORMERS"

    # LoRA.
    lora_r: int = 128
    lora_alpha: int = 256
    lora_dropout: float = 0.0
    lora_target_modules: Tuple[str, ...] = (
        "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
    )
    keep_last_adapters: int = 1

    # GRPO.
    algorithm_name: str = "fepo-wrong-only-kl-failure-sft"
    solver_adapter_name: str = "solver"
    failure_adapter_name: str = "failure"
    num_generations: int = 8
    grpo_adv_eps: float = 1e-8
    fepo_escape_alpha: float = 0.01
    fepo_reference_beta: float = 0.03
    failure_learning_rate: float = 1.25e-6
    failure_clip_eps: float = 0.2
    failure_kl_beta: float = 0.0

    # Token-level clipped GRPO update.
    gradient_accumulation_steps: int = 4
    train_prompt_batch_size: int = 2
    max_optimizer_steps: int = 3000
    learning_rate: float = 5e-6
    weight_decay: float = 0.0
    adam_beta1: float = 0.9
    adam_beta2: float = 0.99
    max_grad_norm: float = 1.0
    clip_eps: float = 0.2
    kl_beta: float = 0.0
    token_log_ratio_clip: float = 8.0
    stop_on_nonfinite_grad: bool = True

    # Generation.
    temperature: float = 1.0
    top_p: float = 0.95
    request_timeout_s: int = 240

    # Evaluation generation.
    eval_temperature: float = 1.0
    eval_top_p: float = 0.95

    # Evaluation/logging. eval_generations is k.
    eval_max_examples: Optional[int] = None
    eval_every_steps: int = 20
    eval_generations: int = 8
    eval_batch_size: int = 4
    print_table_every_prompt: bool = False
    print_optimizer_summary_every_step: bool = True
    log_problem_chars: int = 20
    log_response_chars: int = 20
    nvidia_smi_every_steps: int = 25
    run_smoke_test: bool = True
    run_eval_smoke_test: bool = True
    run_full_training: bool = True
    run_final_eval: bool = True

    # Full-state checkpoint/resume.
    checkpoint_enabled: bool = True
    checkpoint_dir_name: str = "fepo_dapo_math_17k_failure_sft_checkpoints"
    checkpoint_every_optimizer_steps: int = 10
    checkpoint_every_accumulation: bool = False
    resume_from_checkpoint: str = None
    keep_last_checkpoints: int = 1

    # Weights & Biases logging.
    wandb_enabled: bool = True
    wandb_project: str = "failure-escape"
    wandb_entity: Optional[str] = None
    wandb_run_name: str = "qwen25-fepo-dapo-math-17k-failure-sft"
    wandb_group: str = "fepo-dapo-math-17k-failure-sft"
    wandb_tags: Tuple[str, ...] = (
        "fepo-wrong-only-kl-failure-sft",
        "fepo-dapo-math-17k-failure-sft",
        "dapo-math-17k",
        "dr-grpo",
        "failure-escape",
        "failure-sft",
        "ground-truth-reward",
        "amc23-eval",
        "grpo",
    )
    wandb_mode: str = "online"
    wandb_run_id: Optional[str] = None
    wandb_resume: Optional[str] = None
    wandb_job_type: str = "train"
    wandb_notes: Optional[str] = "FEPO solver with wrong-only KLs and wrong-only SFT failure adapter on DAPO-Math-17K with AMC23 pass@k evaluation."
    wandb_save_code: bool = False
    wandb_log_tables: bool = False
    wandb_log_model_artifact: bool = False

    # Output paths.
    working_dir: str = "/kaggle/working"


CONFIG = GRPOConfig()

if int(CONFIG.num_generations) < 2:
    raise ValueError("GRPO needs at least two generations per prompt for group normalization.")
if int(CONFIG.num_generations) > int(CONFIG.max_num_seqs):
    raise ValueError("CONFIG.num_generations must be <= CONFIG.max_num_seqs for the current vLLM settings.")
if int(CONFIG.eval_generations) < 1:
    raise ValueError("CONFIG.eval_generations must be at least 1.")
if int(CONFIG.eval_batch_size) < 1:
    raise ValueError("CONFIG.eval_batch_size must be at least 1.")
if int(CONFIG.train_prompt_batch_size) < 1:
    raise ValueError("CONFIG.train_prompt_batch_size must be at least 1.")
if int(CONFIG.train_prompt_batch_size) * int(CONFIG.num_generations) > int(CONFIG.max_num_seqs):
    print(
        "WARNING: train_prompt_batch_size * num_generations exceeds max_num_seqs; "
        "vLLM will queue extra sequences, which may reduce the expected speedup."
    )

random.seed(CONFIG.seed)
np.random.seed(CONFIG.seed)
torch.manual_seed(CONFIG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG.seed)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

WORKING_DIR = Path(CONFIG.working_dir)
ADAPTER_ROOT = WORKING_DIR / "fepo_dapo_math_17k_failure_sft_solver_lora_adapters"
FAILURE_ADAPTER_ROOT = WORKING_DIR / "fepo_dapo_math_17k_failure_sft_failure_lora_adapters"
CHECKPOINT_ROOT = WORKING_DIR / CONFIG.checkpoint_dir_name
LATEST_CHECKPOINT_PATH = CHECKPOINT_ROOT / "latest_checkpoint.json"
LOG_PATH = WORKING_DIR / "fepo_dapo_math_17k_failure_sft_logs.jsonl"
FINAL_ADAPTER_DIR = WORKING_DIR / "qwen25_15b_fepo_dapo_math_17k_failure_sft_solver_lora_final"
FINAL_FAILURE_ADAPTER_DIR = WORKING_DIR / "qwen25_15b_fepo_dapo_math_17k_failure_sft_failure_lora_final"
VLLM_LOG_PATH = WORKING_DIR / "vllm_server.log"

for p in [WORKING_DIR, ADAPTER_ROOT, FAILURE_ADAPTER_ROOT, CHECKPOINT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

KAGGLE_RESUME_INPUT_DIR = Path("/kaggle/input/models/swapnil776/train-274/transformers/default/1")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_RESUME_ITEMS = (
    CONFIG.checkpoint_dir_name,
    ADAPTER_ROOT.name,
    FAILURE_ADAPTER_ROOT.name,
    LOG_PATH.name,
)


def discover_kaggle_resume_input_dir() -> Optional[Path]:
    candidates: List[Path] = []
    if KAGGLE_RESUME_INPUT_DIR.exists():
        candidates.append(KAGGLE_RESUME_INPUT_DIR)
    if KAGGLE_INPUT_ROOT.exists():
        for latest in KAGGLE_INPUT_ROOT.rglob(f"{CONFIG.checkpoint_dir_name}/latest_checkpoint.json"):
            candidates.append(latest.parent.parent)

    unique: List[Path] = []
    seen = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique.append(candidate)

    if not unique:
        return None

    def step_from_pointer(candidate: Path) -> int:
        pointer = candidate / CONFIG.checkpoint_dir_name / "latest_checkpoint.json"
        try:
            payload = json.loads(pointer.read_text(encoding="utf-8"))
            return int(payload.get("optimizer_step", -1))
        except Exception:
            return -1

    return sorted(unique, key=step_from_pointer, reverse=True)[0]


def restore_kaggle_resume_artifacts() -> None:
    if not bool(CONFIG.checkpoint_enabled):
        return
    requested = str(CONFIG.resume_from_checkpoint or "").strip()
    if requested not in {"latest", "auto"}:
        return
    if LATEST_CHECKPOINT_PATH.exists():
        print("Found existing latest checkpoint pointer:", LATEST_CHECKPOINT_PATH)
        return

    src_root = discover_kaggle_resume_input_dir()
    if src_root is None:
        print("No FEPO DAPO-Math-17K Dr.GRPO checkpoint artifacts found under /kaggle/input for resume.")
        return

    print("Restoring FEPO DAPO-Math-17K Dr.GRPO resume artifacts from:", src_root)
    for item in KAGGLE_RESUME_ITEMS:
        src = src_root / item
        dst = WORKING_DIR / item
        if not src.exists():
            print("Resume artifact missing:", src)
            continue
        if dst.exists():
            if dst.is_dir():
                shutil.rmtree(dst)
            else:
                dst.unlink()
        if src.is_dir():
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        print("Copied", src, "->", dst)


SYSTEM_PROMPT = (
    "You are a careful mathematical problem solver. "
    "Reason step by step and put the final answer in \\boxed{}."
)
USER_SUFFIX = "Solve the problem. Show your reasoning and put the final answer in \\boxed{}."

restore_kaggle_resume_artifacts()


def repair_latest_checkpoint_pointer(latest_path: Path) -> None:
    latest_path = Path(latest_path)
    if not latest_path.exists():
        return
    payload = json.loads(latest_path.read_text(encoding="utf-8"))
    raw_path = payload.get("checkpoint_path") or payload.get("path")
    current = Path(raw_path) if raw_path else None
    if current is not None and current.exists() and (current / "training_state.pt").exists() and (current / "adapter").exists():
        return

    candidates: List[Path] = []
    if current is not None:
        candidates.append(latest_path.parent / current.name)
    candidates.extend(sorted(latest_path.parent.glob("checkpoint_step_*_optimizer"), key=lambda p: p.name, reverse=True))
    candidates.extend(sorted(latest_path.parent.glob("checkpoint_step_*_accum_*"), key=lambda p: p.name, reverse=True))

    for candidate in candidates:
        if candidate.exists() and (candidate / "training_state.pt").exists() and (candidate / "adapter").exists():
            payload["checkpoint_path"] = str(candidate)
            payload["path"] = str(candidate)
            payload["adapter_path"] = str(candidate / "adapter")
            tmp = latest_path.with_suffix(".json.tmp")
            tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
            tmp.replace(latest_path)
            print("Repaired latest checkpoint pointer to:", candidate)
            return


def remap_checkpoint_path_from_input(checkpoint_path: Path) -> Optional[Path]:
    checkpoint_path = Path(checkpoint_path)
    if checkpoint_path.exists():
        return checkpoint_path
    parts = checkpoint_path.parts
    if CONFIG.checkpoint_dir_name in parts:
        idx = parts.index(CONFIG.checkpoint_dir_name)
        relative_checkpoint = Path(*parts[idx:])
        if KAGGLE_INPUT_ROOT.exists():
            for latest in KAGGLE_INPUT_ROOT.rglob(f"{CONFIG.checkpoint_dir_name}/latest_checkpoint.json"):
                candidate = latest.parent.parent / relative_checkpoint
                if candidate.exists():
                    print("Remapped checkpoint path from working to input:", checkpoint_path, "->", candidate)
                    return candidate
    return None


def resolve_resume_checkpoint_path() -> Optional[Path]:
    requested = str(getattr(CONFIG, "resume_from_checkpoint", "") or "").strip()
    if not requested or not bool(getattr(CONFIG, "checkpoint_enabled", False)):
        return None

    if requested in {"latest", "auto"}:
        repair_latest_checkpoint_pointer(LATEST_CHECKPOINT_PATH)
        if not LATEST_CHECKPOINT_PATH.exists():
            restored = discover_kaggle_resume_input_dir()
            if restored is not None:
                latest_in_input = restored / CONFIG.checkpoint_dir_name / "latest_checkpoint.json"
                if latest_in_input.exists():
                    payload = json.loads(latest_in_input.read_text(encoding="utf-8"))
                    requested_path = payload.get("checkpoint_path") or payload.get("path")
                    if requested_path:
                        remapped = remap_checkpoint_path_from_input(Path(requested_path))
                        if remapped is not None:
                            return remapped
            raise FileNotFoundError(
                "CONFIG.resume_from_checkpoint='latest' but no latest checkpoint pointer was found."
            )
        try:
            payload = json.loads(LATEST_CHECKPOINT_PATH.read_text(encoding="utf-8"))
        except Exception as exc:
            raise RuntimeError(f"Could not read latest checkpoint pointer {LATEST_CHECKPOINT_PATH}: {exc}") from exc
        checkpoint_path = payload.get("checkpoint_path") or payload.get("path")
        if not checkpoint_path:
            raise RuntimeError(f"Latest checkpoint pointer {LATEST_CHECKPOINT_PATH} has no checkpoint_path/path field.")
        resolved = Path(checkpoint_path)
    else:
        resolved = Path(requested)

    if resolved.exists():
        return resolved
    remapped = remap_checkpoint_path_from_input(resolved)
    if remapped is not None:
        return remapped
    raise FileNotFoundError(
        f"Requested checkpoint path does not exist: {resolved}. Stop this run instead of starting from step 1."
    )


def checkpoint_state_path(checkpoint_dir: Path) -> Path:
    return Path(checkpoint_dir) / "training_state.pt"


def checkpoint_metadata_path(checkpoint_dir: Path) -> Path:
    return Path(checkpoint_dir) / "metadata.json"


def checkpoint_adapter_path(checkpoint_dir: Path) -> Path:
    return Path(checkpoint_dir) / "adapter"


def read_checkpoint_metadata(checkpoint_dir: Optional[Path]) -> Dict[str, Any]:
    if checkpoint_dir is None:
        return {}
    metadata_path = checkpoint_metadata_path(checkpoint_dir)
    if not metadata_path.exists():
        return {}
    try:
        return json.loads(metadata_path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Could not read checkpoint metadata {metadata_path}: {exc}")
        return {}


RESUME_CHECKPOINT_PATH = resolve_resume_checkpoint_path()
RESUME_CHECKPOINT_METADATA = read_checkpoint_metadata(RESUME_CHECKPOINT_PATH)
if RESUME_CHECKPOINT_PATH is not None:
    print("Resume checkpoint selected:", RESUME_CHECKPOINT_PATH)
    checkpoint_wandb_id = str(RESUME_CHECKPOINT_METADATA.get("wandb_run_id") or "").strip()
    if checkpoint_wandb_id and not (os.environ.get("WANDB_RUN_ID") or CONFIG.wandb_run_id):
        CONFIG.wandb_run_id = checkpoint_wandb_id
    if checkpoint_wandb_id and not (os.environ.get("WANDB_RESUME") or CONFIG.wandb_resume):
        CONFIG.wandb_resume = "allow"

# Set WANDB_API_KEY through Kaggle Secrets or os.environ. Do not paste tokens into this notebook.
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "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")


def get_wandb_api_key() -> str:
    return WANDB_API_KEY


def grpo_response_count() -> int:
    return int(CONFIG.num_generations)


def _wandb_run_name() -> str:
    if CONFIG.wandb_run_name:
        return CONFIG.wandb_run_name
    model_short = CONFIG.model_id.split("/")[-1]
    return (
        f"{CONFIG.algorithm_name}_{model_short}_{CONFIG.train_dataset_name}"
        f"_g{grpo_response_count()}_lr{CONFIG.learning_rate}"
    ).replace("/", "-")


def _wandb_config_payload() -> Dict[str, Any]:
    payload = asdict(CONFIG)
    payload.update({
        "algorithm": CONFIG.algorithm_name,
        "response_count": grpo_response_count(),
        "responses_per_optimizer_step": grpo_response_count() * CONFIG.gradient_accumulation_steps,
        "adapter_root": str(ADAPTER_ROOT),
        "checkpoint_root": str(CHECKPOINT_ROOT),
        "resume_checkpoint_path": str(RESUME_CHECKPOINT_PATH) if RESUME_CHECKPOINT_PATH is not None else None,
        "log_path": str(LOG_PATH),
        "final_adapter_dir": str(FINAL_ADAPTER_DIR),
        "training_reward_source": "ground_truth_answer_match",
        "training_objective": "fepo_point_kl",
        "failure_adapter_root": str(FAILURE_ADAPTER_ROOT),
        "final_failure_adapter_dir": str(FINAL_FAILURE_ADAPTER_DIR),
        "evaluation_metric": "pass@1/pass@k/avg@k",
    })
    return payload


def init_wandb_run():
    if not bool(CONFIG.wandb_enabled):
        print("W&B disabled by config.")
        return None
    if wandb is None:
        print("W&B import failed or is unavailable; continuing without W&B.")
        return None
    api_key = get_wandb_api_key()
    if api_key:
        try:
            wandb.login(key=api_key, relogin=True)
        except Exception as exc:
            print("W&B login failed; continuing without W&B:", exc)
            return None
    elif CONFIG.wandb_mode != "offline":
        print("WANDB_API_KEY is not set. Set CONFIG.wandb_mode='offline' or provide a key for online logging.")

    init_kwargs = {
        "project": CONFIG.wandb_project,
        "entity": CONFIG.wandb_entity,
        "name": _wandb_run_name(),
        "group": CONFIG.wandb_group,
        "job_type": CONFIG.wandb_job_type,
        "tags": list(CONFIG.wandb_tags),
        "notes": CONFIG.wandb_notes,
        "config": _wandb_config_payload(),
        "mode": CONFIG.wandb_mode,
        "save_code": CONFIG.wandb_save_code,
    }
    run_id = os.environ.get("WANDB_RUN_ID") or CONFIG.wandb_run_id
    resume = os.environ.get("WANDB_RESUME") or CONFIG.wandb_resume
    if run_id:
        init_kwargs["id"] = run_id
    if resume:
        init_kwargs["resume"] = resume

    try:
        run = wandb.init(**init_kwargs)
        print("W&B run:", run.name, run.id)
        return run
    except Exception as exc:
        print("W&B init failed; continuing without W&B:", exc)
        return None


WANDB_RUN = init_wandb_run()


## 3. DAPO-Math-17K training data, AMC23 eval data, answer extraction, and unit tests

DAPO-Math-17K is prepared for FEPO training rewards from ground-truth labels. AMC23 is prepared only for evaluation.


In [ ]:
def strip_latex_wrappers(text: str) -> str:
    s = str(text).strip()
    s = s.replace("$", "")
    s = re.sub(r"\\left|\\right", "", s)
    s = re.sub(r"\\,|\\;|\\!|\\ ", "", s)
    s = s.strip()
    while len(s) > 1 and s[-1] in ".,;:":
        if s[-1] == "." and re.search(r"\d\.\d*$", s):
            break
        s = s[:-1].strip()
    return s


def decimal_to_canonical(value: Decimal) -> str:
    text = format(value, "f")
    if "." in text:
        text = text.rstrip("0").rstrip(".")
    if text in {"", "-0"}:
        return "0"
    return text


def try_decimal(text: str) -> Optional[Decimal]:
    s = strip_latex_wrappers(text)
    s = s.replace(",", "").replace("_", "")
    s = re.sub(r"^(USD|usd)\s*", "", s)
    s = s.strip()
    if s.startswith("\\text{") and s.endswith("}"):
        s = s[len("\\text{"):-1].strip()

    frac_match = re.fullmatch(r"\\frac\{([^{}]+)\}\{([^{}]+)\}", s)
    if frac_match:
        num = try_decimal(frac_match.group(1))
        den = try_decimal(frac_match.group(2))
        if num is not None and den not in (None, Decimal(0)):
            return num / den

    simple_frac = re.fullmatch(r"([-+]?\d+(?:\.\d+)?)/([-+]?\d+(?:\.\d+)?)", s)
    if simple_frac:
        den = Decimal(simple_frac.group(2))
        if den != 0:
            return Decimal(simple_frac.group(1)) / den

    percent_match = re.fullmatch(r"([-+]?\d+(?:\.\d+)?)%", s)
    if percent_match:
        return Decimal(percent_match.group(1)) / Decimal(100)

    if re.fullmatch(r"[-+]?\d+(?:\.\d+)?", s):
        try:
            return Decimal(s)
        except InvalidOperation:
            return None
    return None


def normalize_answer_text(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    s = strip_latex_wrappers(str(text))
    if not s:
        return None
    direct_value = try_decimal(s)
    if direct_value is not None:
        return decimal_to_canonical(direct_value)
    s = s.replace("\\boxed", "")
    s = re.sub(r"\\(?:mathrm|text)\{([^{}]*)\}", r"\1", s)
    s = s.strip("{}()[] ")
    s = re.sub(r"\s+", " ", s).strip()
    value = try_decimal(s)
    if value is not None:
        return decimal_to_canonical(value)
    s = s.replace(",", "")
    s = s.strip().lower()
    while len(s) > 1 and s[-1] in ".,;:":
        s = s[:-1].strip()
    return s or None


def extract_all_boxed(text: str) -> List[str]:
    text = str(text)
    results = []
    marker = r"\boxed{"
    start = 0
    while True:
        idx = text.find(marker, start)
        if idx == -1:
            break
        pos = idx + len(marker)
        depth = 1
        chars = []
        while pos < len(text) and depth > 0:
            ch = text[pos]
            if ch == "{":
                depth += 1
                chars.append(ch)
            elif ch == "}":
                depth -= 1
                if depth > 0:
                    chars.append(ch)
            else:
                chars.append(ch)
            pos += 1
        if depth == 0:
            results.append("".join(chars).strip())
        start = pos
    return results


def extract_last_boxed(text: str) -> Optional[str]:
    boxes = extract_all_boxed(text)
    return boxes[-1] if boxes else None


def extract_after_hash_answer(text: str) -> Optional[str]:
    match = re.search(r"####\s*(.+)$", str(text), flags=re.DOTALL)
    if not match:
        return None
    return match.group(1).strip()


def extract_final_number_or_text(text: str) -> Optional[str]:
    s = str(text).strip()
    if not s:
        return None
    number_matches = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?(?:/[-+]?\d[\d,]*(?:\.\d+)?)?%?", s)
    if number_matches:
        return number_matches[-1]
    lines = [line.strip() for line in s.splitlines() if line.strip()]
    if not lines:
        return None
    return lines[-1]


def extract_model_answer(text: str) -> Optional[str]:
    boxed = extract_last_boxed(text)
    if boxed is not None:
        return boxed
    hashed = extract_after_hash_answer(text)
    if hashed is not None:
        return hashed
    return extract_final_number_or_text(text)


def normalize_model_answer(text: str) -> Optional[str]:
    return normalize_answer_text(extract_model_answer(text))


def normalize_ground_truth_answer(answer: Any, dataset_kind: str) -> Optional[str]:
    if dataset_kind in {"dapo-math-17k", "amc23"}:
        candidate = str(answer)
    else:
        candidate = extract_after_hash_answer(str(answer)) or extract_final_number_or_text(str(answer))
    return normalize_answer_text(candidate)


def make_messages(problem: str) -> List[Dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{problem}\n{USER_SUFFIX}"},
    ]


def load_dataset_with_optional_config(name: str, config: Optional[str]):
    if config is None:
        return load_dataset(name)
    return load_dataset(name, config)


DAPO_PROMPT_INTRO = "Solve the following math problem step by step."
DAPO_PROMPT_REMINDER_RE = re.compile(
    r'\n\nRemember to put your answer on its own line after "Answer:"\.?\s*$',
    flags=re.IGNORECASE,
)


def extract_dapo_problem(prompt: Any) -> str:
    if isinstance(prompt, (list, tuple)):
        messages = list(prompt)
        user_messages = [
            message for message in messages
            if isinstance(message, dict) and message.get("role") == "user"
        ]
        message = user_messages[0] if user_messages else (messages[0] if messages else {})
        content = message.get("content", "") if isinstance(message, dict) else str(message)
    elif isinstance(prompt, dict):
        content = prompt.get("content", "")
    else:
        content = str(prompt)

    problem = str(content).strip()
    if problem.startswith(DAPO_PROMPT_INTRO) and "\n\n" in problem:
        problem = problem.split("\n\n", 1)[1].strip()
    problem = DAPO_PROMPT_REMINDER_RE.sub("", problem).strip()
    if not problem:
        raise ValueError(f"Could not extract DAPO-Math-17K problem from prompt: {prompt!r}")
    return problem


def prepare_dapo_math_split(split) -> List[Dict[str, Any]]:
    required = {"prompt", "label"}
    missing = required.difference(split.column_names)
    if missing:
        raise KeyError(f"DAPO-Math-17K split is missing expected columns: {sorted(missing)}. Found: {split.column_names}")
    examples = []
    for idx, row in enumerate(split):
        raw_prompt = row["prompt"]
        problem = extract_dapo_problem(raw_prompt)
        raw_answer = row["label"]
        target = normalize_ground_truth_answer(raw_answer, "dapo-math-17k")
        if target is None:
            raise ValueError(f"Could not normalize DAPO-Math-17K label for row {idx}: {raw_answer!r}")
        examples.append({
            "dataset": "dapo-math-17k",
            "example_id": row.get("id", idx) if hasattr(row, "get") else idx,
            "problem": problem,
            "ground_truth_raw": raw_answer,
            "ground_truth_normalized": target,
            "url": None,
            "messages": make_messages(problem),
        })
    return examples


def prepare_amc23_split(split) -> List[Dict[str, Any]]:
    required = {"question", "answer"}
    missing = required.difference(split.column_names)
    if missing:
        raise KeyError(f"AMC23 split is missing expected columns: {sorted(missing)}. Found: {split.column_names}")
    examples = []
    for idx, row in enumerate(split):
        problem = row["question"]
        raw_answer = row["answer"]
        target = normalize_ground_truth_answer(raw_answer, "amc23")
        if target is None:
            raise ValueError(f"Could not normalize AMC23 answer for row {idx}: {raw_answer!r}")
        examples.append({
            "dataset": "amc23",
            "example_id": row.get("id", idx) if hasattr(row, "get") else idx,
            "problem": problem,
            "ground_truth_raw": raw_answer,
            "ground_truth_normalized": target,
            "url": row.get("url") if hasattr(row, "get") else None,
            "messages": make_messages(problem),
        })
    return examples


def run_answer_extraction_unit_tests() -> None:
    assert normalize_answer_text("1,234") == "1234"
    assert normalize_answer_text(r"\frac{1}{2}") == "0.5"
    assert normalize_ground_truth_answer("Natalia did it. #### 72", "legacy_hash") == "72"
    assert normalize_ground_truth_answer("34", "dapo-math-17k") == "34"
    assert normalize_ground_truth_answer("-3", "dapo-math-17k") == "-3"
    assert normalize_ground_truth_answer("1 or 4", "dapo-math-17k") == "1 or 4"
    assert normalize_ground_truth_answer("3159", "amc23") == "3159"
    assert normalize_model_answer(r"We get \boxed{204}.") == "204"
    assert normalize_model_answer("Reasoning... #### 18") == "18"
    assert normalize_model_answer("The answer is 27.") == "27"
    print("Answer extraction tests passed.")


def run_dapo_prompt_extraction_unit_tests() -> None:
    sample_prompt = [{
        "role": "user",
        "content": (
            "Solve the following math problem step by step. The last line of your response "
            "should be of the form Answer: \\boxed{$Answer} where $Answer is the answer to the problem.\n\n"
            "In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$. Find $AB+AC$.\n\n"
            "Remember to put your answer on its own line after \"Answer:\"."
        ),
    }]
    problem = extract_dapo_problem(sample_prompt)
    assert problem.startswith("In triangle $ABC$")
    assert "Remember to put your answer" not in problem
    messages = make_messages(problem)
    assert messages[0]["role"] == "system"
    assert messages[1]["role"] == "user"
    assert messages[1]["content"].endswith(USER_SUFFIX)
    print("DAPO prompt extraction tests passed.")


run_answer_extraction_unit_tests()
run_dapo_prompt_extraction_unit_tests()

train_raw_ds = load_dataset_with_optional_config(CONFIG.train_dataset_name, CONFIG.train_dataset_config)
if CONFIG.train_split not in train_raw_ds:
    available = list(train_raw_ds.keys())
    raise KeyError(f"Split {CONFIG.train_split!r} not found in {CONFIG.train_dataset_name}. Available splits: {available}")

eval_raw_ds = load_dataset_with_optional_config(CONFIG.eval_dataset_name, CONFIG.eval_dataset_config)
if CONFIG.eval_split not in eval_raw_ds:
    available = list(eval_raw_ds.keys())
    raise KeyError(f"Split {CONFIG.eval_split!r} not found in {CONFIG.eval_dataset_name}. Available splits: {available}")

train_examples = prepare_dapo_math_split(train_raw_ds[CONFIG.train_split])
if CONFIG.train_example_limit is not None:
    train_examples = train_examples[: CONFIG.train_example_limit]
eval_examples = prepare_amc23_split(eval_raw_ds[CONFIG.eval_split])

train_sample = train_examples[0]
eval_sample = eval_examples[0]
assert {"example_id", "problem", "ground_truth_raw", "ground_truth_normalized", "messages"}.issubset(train_sample)
assert {"example_id", "problem", "ground_truth_raw", "ground_truth_normalized", "messages"}.issubset(eval_sample)

print(f"train dataset: {CONFIG.train_dataset_name}/{CONFIG.train_dataset_config or 'default'} split={CONFIG.train_split}")
print(f"train examples: {len(train_examples)}")
print("train columns:", train_raw_ds[CONFIG.train_split].column_names)
print("train sample problem:", train_sample["problem"][:300])
print("train sample target:", train_sample["ground_truth_normalized"])
print(f"eval dataset: {CONFIG.eval_dataset_name}/{CONFIG.eval_dataset_config or 'default'} split={CONFIG.eval_split}")
print(f"eval examples: {len(eval_examples)}")
print("eval columns:", eval_raw_ds[CONFIG.eval_split].column_names)
print("eval sample problem:", eval_sample["problem"][:300])
print("eval sample target:", eval_sample["ground_truth_normalized"])


## 4. vLLM server helpers

The server runs in a subprocess with `CUDA_VISIBLE_DEVICES=0`, so inside the vLLM process its GPU is `cuda:0`. The notebook kernel keeps access to both GPUs and uses `cuda:1` for LoRA training, frozen-reference KL scoring, and old-policy logprob scoring.


In [ ]:
VLLM_BASE_URL = f"http://{CONFIG.vllm_host}:{CONFIG.vllm_port}/v1"
vllm_process: Optional[subprocess.Popen] = None


def vllm_cmd(enforce_eager: Optional[bool] = None) -> List[str]:
    if enforce_eager is None:
        enforce_eager = bool(CONFIG.vllm_enforce_eager)
    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", CONFIG.model_id,
        "--served-model-name", CONFIG.vllm_base_model_name,
        "--host", CONFIG.vllm_host,
        "--port", str(CONFIG.vllm_port),
        "--dtype", "float16",
        "--max-model-len", str(CONFIG.max_model_len),
        "--max-num-seqs", str(CONFIG.max_num_seqs),
        "--gpu-memory-utilization", str(CONFIG.vllm_gpu_memory_utilization),
        "--enable-lora",
        "--max-lora-rank", str(CONFIG.lora_r),
        "--max-loras", "1",
        "--disable-log-requests",
    ]
    if enforce_eager:
        cmd.append("--enforce-eager")
    return cmd


def vllm_launch_env() -> Dict[str, str]:
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = CONFIG.vllm_visible_device
    env["VLLM_ALLOW_RUNTIME_LORA_UPDATING"] = "True"
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    backend = str(getattr(CONFIG, "vllm_attention_backend", "") or "").strip()
    if backend and backend.lower() not in {"auto", "default", "none"}:
        env["VLLM_ATTENTION_BACKEND"] = backend.upper()
    return env


def vllm_is_ready() -> bool:
    try:
        response = requests.get(f"{VLLM_BASE_URL}/models", timeout=5)
        return response.status_code == 200
    except requests.RequestException:
        return False


def wait_for_vllm(timeout_s: int = 600) -> None:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if vllm_is_ready():
            print("vLLM server is ready.")
            return
        if vllm_process is not None and vllm_process.poll() is not None:
            tail = VLLM_LOG_PATH.read_text(errors="ignore")[-4000:] if VLLM_LOG_PATH.exists() else ""
            raise RuntimeError(f"vLLM exited early with code {vllm_process.returncode}. Log tail:\n{tail}")
        time.sleep(5)
    tail = VLLM_LOG_PATH.read_text(errors="ignore")[-4000:] if VLLM_LOG_PATH.exists() else ""
    raise TimeoutError(f"vLLM did not become ready within {timeout_s}s. Log tail:\n{tail}")


def _terminate_tracked_vllm(timeout_s: int = 20) -> None:
    global vllm_process
    if vllm_process is None or vllm_process.poll() is not None:
        vllm_process = None
        return
    vllm_process.terminate()
    try:
        vllm_process.wait(timeout=timeout_s)
    except subprocess.TimeoutExpired:
        vllm_process.kill()
        vllm_process.wait(timeout=timeout_s)
    vllm_process = None


def _start_vllm(enforce_eager: bool) -> subprocess.Popen:
    env = vllm_launch_env()
    cmd = vllm_cmd(enforce_eager=enforce_eager)
    VLLM_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(VLLM_LOG_PATH, "a", buffering=1)
    mode = "eager fallback" if enforce_eager else "optimized"
    print(
        f"Launching vLLM ({mode}, attention_backend={env.get('VLLM_ATTENTION_BACKEND', 'auto')}):",
        " ".join(cmd),
    )
    process = subprocess.Popen(
        cmd,
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )
    return process


def launch_vllm() -> Optional[subprocess.Popen]:
    global vllm_process
    if vllm_is_ready():
        print(
            "Found an existing vLLM server at", VLLM_BASE_URL,
            "- stop/restart it to apply vLLM backend or eager-mode config changes.",
        )
        return vllm_process

    enforce_eager = bool(CONFIG.vllm_enforce_eager)
    vllm_process = _start_vllm(enforce_eager=enforce_eager)
    try:
        wait_for_vllm()
        return vllm_process
    except Exception as exc:
        if enforce_eager or not bool(getattr(CONFIG, "vllm_allow_eager_fallback", True)):
            raise
        print("Optimized vLLM launch failed; retrying with --enforce-eager. Error:", exc)
        _terminate_tracked_vllm()
        vllm_process = _start_vllm(enforce_eager=True)
        wait_for_vllm()
        return vllm_process


def post_vllm(path: str, payload: Dict[str, Any], timeout_s: int = 120) -> requests.Response:
    return requests.post(f"{VLLM_BASE_URL}{path}", json=payload, timeout=timeout_s)


def get_vllm_models() -> Dict[str, Any]:
    response = requests.get(f"{VLLM_BASE_URL}/models", timeout=30)
    response.raise_for_status()
    return response.json()


def unload_lora_adapter(lora_name: str, ignore_missing: bool = True) -> None:
    response = post_vllm("/unload_lora_adapter", {"lora_name": lora_name}, timeout_s=120)
    if response.status_code == 200:
        print(f"Unloaded LoRA adapter {lora_name}.")
        return
    if ignore_missing and response.status_code in {400, 404, 409}:
        print(f"LoRA adapter {lora_name} was not loaded or could not be unloaded cleanly: {response.text[:200]}")
        return
    response.raise_for_status()


def load_lora_adapter(lora_name: str, lora_path: Path) -> None:
    payload = {"lora_name": lora_name, "lora_path": str(lora_path)}
    response = post_vllm("/load_lora_adapter", payload, timeout_s=180)
    if response.status_code != 200:
        raise RuntimeError(f"Failed to load LoRA adapter: {response.status_code} {response.text}")
    print(f"Loaded LoRA adapter {lora_name} from {lora_path}.")


def generate_with_vllm(
    messages: List[Dict[str, str]],
    n: int,
    temperature: float,
    top_p: float,
    max_tokens: int,
    timeout_s: Optional[int] = None,
) -> List[str]:
    payload = {
        "model": CONFIG.vllm_lora_name,
        "messages": messages,
        "n": n,
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
    }
    response = post_vllm("/chat/completions", payload, timeout_s=timeout_s or CONFIG.request_timeout_s)
    if response.status_code != 200:
        log_tail = VLLM_LOG_PATH.read_text(errors="ignore")[-4000:] if VLLM_LOG_PATH.exists() else ""
        raise RuntimeError(
            f"vLLM generation failed: {response.status_code} {response.text}\n"
            f"Model requested: {payload.get('model')}\n"
            f"vLLM log tail:\n{log_tail}"
        )
    data = response.json()
    choices = data.get("choices", [])
    if len(choices) != int(n):
        print(f"WARNING: vLLM returned {len(choices)} choices for requested n={n}.")
    return [choice["message"]["content"] for choice in choices]


def print_nvidia_smi() -> None:
    try:
        result = subprocess.run(["nvidia-smi"], text=True, capture_output=True, check=False)
        print(result.stdout[-4000:])
    except FileNotFoundError:
        print("nvidia-smi is not available in this environment.")


## 5. Load policy, frozen reference model, LoRA adapters, and optimizers

The training-side Transformers model owns two named LoRA adapters: a solver adapter and a failure adapter. Each adapter has its own optimizer.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for the Kaggle two-GPU training run.")

if torch.cuda.device_count() < 2:
    raise RuntimeError("This notebook expects two GPUs. Enable the Kaggle two-T4 accelerator.")

TRAIN_DEVICE = torch.device(CONFIG.train_device)
torch.cuda.set_device(TRAIN_DEVICE)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG.model_id, trust_remote_code=False, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SPECIAL_TOKEN_IDS = set(tokenizer.all_special_ids or [])
PAD_TOKEN_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

print("Loading trainable FEPO policy on", TRAIN_DEVICE)
base_policy_model = AutoModelForCausalLM.from_pretrained(
    CONFIG.model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=False,
    attn_implementation="sdpa",
)
base_policy_model.to(TRAIN_DEVICE)
base_policy_model.config.use_cache = False
base_policy_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

peft_config = LoraConfig(
    r=CONFIG.lora_r,
    lora_alpha=CONFIG.lora_alpha,
    target_modules=list(CONFIG.lora_target_modules),
    lora_dropout=CONFIG.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)
policy_model = get_peft_model(base_policy_model, peft_config, adapter_name=CONFIG.solver_adapter_name)
policy_model.add_adapter(CONFIG.failure_adapter_name, peft_config)
if hasattr(policy_model, "enable_input_require_grads"):
    policy_model.enable_input_require_grads()


def _adapter_name_marker(adapter_name: str) -> str:
    return f".{adapter_name}."


def adapter_named_parameters(adapter_name: str) -> Dict[str, torch.nn.Parameter]:
    marker = _adapter_name_marker(adapter_name)
    return {name: param for name, param in policy_model.named_parameters() if marker in name}


def set_active_policy_adapter(adapter_name: str, train: bool = True) -> None:
    policy_model.set_adapter(adapter_name)
    if train:
        policy_model.train()
    else:
        policy_model.eval()
    for name, param in policy_model.named_parameters():
        is_selected_adapter = _adapter_name_marker(adapter_name) in name
        is_any_lora_adapter = (
            _adapter_name_marker(CONFIG.solver_adapter_name) in name
            or _adapter_name_marker(CONFIG.failure_adapter_name) in name
        )
        if is_any_lora_adapter:
            param.requires_grad_(bool(train and is_selected_adapter))


set_active_policy_adapter(CONFIG.solver_adapter_name, train=True)
solver_params = list(adapter_named_parameters(CONFIG.solver_adapter_name).values())
failure_params = list(adapter_named_parameters(CONFIG.failure_adapter_name).values())
if not solver_params:
    raise RuntimeError(f"No trainable parameters found for solver adapter {CONFIG.solver_adapter_name!r}.")
if not failure_params:
    raise RuntimeError(f"No trainable parameters found for failure adapter {CONFIG.failure_adapter_name!r}.")

print("Solver adapter trainable parameters:", sum(p.numel() for p in solver_params))
print("Failure adapter trainable parameters:", sum(p.numel() for p in failure_params))

print("Loading frozen reference model on", TRAIN_DEVICE)
reference_model = AutoModelForCausalLM.from_pretrained(
    CONFIG.model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=False,
    attn_implementation="sdpa",
)
reference_model.to(TRAIN_DEVICE)
reference_model.config.use_cache = False
reference_model.eval()
for param in reference_model.parameters():
    param.requires_grad_(False)

solver_optimizer = bnb.optim.AdamW8bit(
    solver_params,
    lr=CONFIG.learning_rate,
    betas=(CONFIG.adam_beta1, CONFIG.adam_beta2),
    weight_decay=CONFIG.weight_decay,
)
failure_optimizer = bnb.optim.AdamW8bit(
    failure_params,
    lr=CONFIG.failure_learning_rate,
    betas=(CONFIG.adam_beta1, CONFIG.adam_beta2),
    weight_decay=CONFIG.weight_decay,
)
solver_optimizer.zero_grad(set_to_none=True)
failure_optimizer.zero_grad(set_to_none=True)

print("FEPO models loaded.")
print_nvidia_smi()


## 6. Tokenization, teacher-forced log-probability, and ground-truth FEPO/Dr.GRPO helpers

Solver and failure advantages are mean-centered within each prompt group. Reward standard deviations are logged for diagnostics only and are not used to scale the loss.


In [ ]:
def prompt_token_ids(messages: List[Dict[str, str]]) -> List[int]:
    ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors=None,
    )
    return ids.tolist() if hasattr(ids, "tolist") else list(ids)


def encode_prompt_response(messages: List[Dict[str, str]], response_text: str) -> Optional[Dict[str, Any]]:
    prompt_ids = prompt_token_ids(messages)
    response_ids = tokenizer(response_text, add_special_tokens=False)["input_ids"]
    if len(response_ids) == 0:
        return None
    available = CONFIG.max_model_len - len(prompt_ids)
    if available <= 0:
        return None
    if len(response_ids) > available:
        response_ids = response_ids[:available]
    input_ids_list = list(prompt_ids) + list(response_ids)
    input_ids = torch.tensor([input_ids_list], dtype=torch.long, device=TRAIN_DEVICE)
    attention_mask = torch.ones_like(input_ids, device=TRAIN_DEVICE)
    return {
        "prompt_ids": prompt_ids,
        "response_ids": response_ids,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "completion_start": len(prompt_ids),
        "completion_length": len(response_ids),
    }


def completion_logprobs(model: torch.nn.Module, input_ids: torch.Tensor, completion_start: int) -> torch.Tensor:
    if input_ids.shape[1] <= completion_start:
        return input_ids.new_zeros((0,), dtype=torch.float32)
    outputs = model(
        input_ids=input_ids,
        attention_mask=torch.ones_like(input_ids),
        use_cache=False,
    )
    logits = outputs.logits[:, :-1, :]
    labels = input_ids[:, 1:]
    token_logps = F.log_softmax(logits, dim=-1).gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    start = max(0, completion_start - 1)
    return token_logps[0, start:].float()


def detached_completion_logprobs(model: torch.nn.Module, encoded: Dict[str, Any]) -> torch.Tensor:
    was_training = model.training
    model.eval()
    with torch.no_grad():
        out = completion_logprobs(model, encoded["input_ids"], encoded["completion_start"])
    if was_training:
        model.train()
    return out.detach().cpu()


def reference_completion_logps_from_logits(logits: torch.Tensor, input_ids: torch.Tensor, completion_start: int) -> torch.Tensor:
    if input_ids.shape[1] <= completion_start:
        return logits.new_zeros((0,), dtype=torch.float32)
    shifted_logits = logits[:, :-1, :]
    labels = input_ids[:, 1:]
    token_logps = F.log_softmax(shifted_logits, dim=-1).gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    start = max(0, completion_start - 1)
    return token_logps[0, start:].float()


def detached_reference_completion_stats(encoded: Dict[str, Any]) -> Dict[str, Any]:
    reference_model.eval()
    with torch.no_grad():
        outputs = reference_model(
            input_ids=encoded["input_ids"],
            attention_mask=encoded["attention_mask"],
            use_cache=False,
        )
        ref_logps = reference_completion_logps_from_logits(
            outputs.logits,
            encoded["input_ids"],
            encoded["completion_start"],
        )
    return {
        "ref_logps": ref_logps.detach().cpu(),
        "ref_logp_sum": float(ref_logps.sum().detach().cpu()) if ref_logps.numel() else -float("inf"),
        "ref_token_count": int(ref_logps.numel()),
    }


def compute_ground_truth_rewards(example: Dict[str, Any], items: Sequence[Dict[str, Any]]) -> None:
    target = example["ground_truth_normalized"]
    for item in items:
        prediction = normalize_model_answer(item.get("response", ""))
        is_correct = prediction is not None and prediction == target
        reward = 1.0 if is_correct else 0.0
        item["prediction_normalized"] = prediction
        item["ground_truth_normalized"] = target
        item["has_parseable_answer"] = prediction is not None
        item["reward"] = reward
        item["is_correct"] = bool(is_correct)
        item["anti_reward"] = 1.0 - 2.0 * reward


def attach_fepo_dr_grpo_group_advantages(items: Sequence[Dict[str, Any]]) -> None:
    if not items:
        return
    rewards = torch.tensor([float(item.get("reward", 0.0)) for item in items], dtype=torch.float32)
    anti_rewards = torch.tensor([float(item.get("anti_reward", 1.0)) for item in items], dtype=torch.float32)
    reward_mean = rewards.mean() if rewards.numel() else torch.tensor(0.0)
    reward_std = rewards.std(unbiased=False) if rewards.numel() else torch.tensor(0.0)  # logged only
    anti_reward_mean = anti_rewards.mean() if anti_rewards.numel() else torch.tensor(0.0)
    anti_reward_std = anti_rewards.std(unbiased=False) if anti_rewards.numel() else torch.tensor(0.0)  # logged only
    solver_advantages = rewards - reward_mean if rewards.numel() else torch.zeros_like(rewards)
    failure_advantages = anti_rewards - anti_reward_mean if anti_rewards.numel() else torch.zeros_like(anti_rewards)

    for idx, item in enumerate(items):
        encoded = item.get("encoded")
        completion_length = int(encoded["completion_length"]) if encoded is not None else 0
        solver_advantage = float(solver_advantages[idx].item()) if idx < solver_advantages.numel() else 0.0
        failure_advantage = float(failure_advantages[idx].item()) if idx < failure_advantages.numel() else 0.0
        failed = float(item.get("reward", 0.0)) == 0.0
        item["grpo_advantage"] = solver_advantage
        item["failure_advantage"] = failure_advantage
        item["group_reward_mean"] = float(reward_mean.item()) if rewards.numel() else 0.0
        item["group_reward_std"] = float(reward_std.item()) if rewards.numel() else 0.0
        item["group_anti_reward_mean"] = float(anti_reward_mean.item()) if anti_rewards.numel() else 0.0
        item["group_anti_reward_std"] = float(anti_reward_std.item()) if anti_rewards.numel() else 0.0
        item["valid_token_count"] = completion_length
        item["ignored_token_count"] = 0
        if completion_length > 0:
            item["token_advantages"] = torch.full((completion_length,), solver_advantage, dtype=torch.float32)
            item["failure_token_advantages"] = torch.full((completion_length,), failure_advantage, dtype=torch.float32)
            item["failed_token_mask"] = torch.full((completion_length,), failed, dtype=torch.bool)
            item["token_loss_mask"] = torch.ones((completion_length,), dtype=torch.bool)
        else:
            item["token_advantages"] = torch.zeros((0,), dtype=torch.float32)
            item["failure_token_advantages"] = torch.zeros((0,), dtype=torch.float32)
            item["failed_token_mask"] = torch.zeros((0,), dtype=torch.bool)
            item["token_loss_mask"] = torch.zeros((0,), dtype=torch.bool)


def run_reward_unit_tests() -> None:
    example = {"ground_truth_normalized": "72"}
    items = [
        {"response": r"We get \boxed{72}."},
        {"response": "Reasoning... #### 72"},
        {"response": r"Wrong \boxed{71}."},
        {"response": "   "},
    ]
    compute_ground_truth_rewards(example, items)
    assert [item["reward"] for item in items] == [1.0, 1.0, 0.0, 0.0]
    assert [item["anti_reward"] for item in items] == [-1.0, -1.0, 1.0, 1.0]
    assert items[0]["prediction_normalized"] == "72"
    assert items[3]["has_parseable_answer"] is False
    print("Ground-truth and anti-reward tests passed.")


def run_fepo_dr_grpo_advantage_unit_tests() -> None:
    items = [
        {"response": "", "reward": 1.0, "anti_reward": -1.0, "encoded": {"completion_length": 4}},
        {"response": "", "reward": 0.0, "anti_reward": 1.0, "encoded": {"completion_length": 4}},
        {"response": "", "reward": 0.0, "anti_reward": 1.0, "encoded": {"completion_length": 4}},
    ]
    attach_fepo_dr_grpo_group_advantages(items)
    solver_advs = torch.tensor([item["grpo_advantage"] for item in items])
    failure_advs = torch.tensor([item["failure_advantage"] for item in items])
    expected_solver = torch.tensor([2.0 / 3.0, -1.0 / 3.0, -1.0 / 3.0], dtype=solver_advs.dtype)
    assert torch.allclose(solver_advs, expected_solver, atol=1e-6)
    assert torch.allclose(failure_advs, -2.0 * solver_advs, atol=1e-6)
    assert abs(float(solver_advs.mean().item())) < 1e-6
    assert items[0]["failed_token_mask"].sum().item() == 0
    assert items[1]["failed_token_mask"].sum().item() == 4

    all_wrong = [
        {"response": "", "reward": 0.0, "anti_reward": 1.0, "encoded": {"completion_length": 3}},
        {"response": "", "reward": 0.0, "anti_reward": 1.0, "encoded": {"completion_length": 3}},
    ]
    attach_fepo_dr_grpo_group_advantages(all_wrong)
    assert all(item["grpo_advantage"] == 0.0 for item in all_wrong)
    assert all(item["failure_advantage"] == 0.0 for item in all_wrong)

    all_correct = [
        {"response": "", "reward": 1.0, "anti_reward": -1.0, "encoded": {"completion_length": 3}},
        {"response": "", "reward": 1.0, "anti_reward": -1.0, "encoded": {"completion_length": 3}},
    ]
    attach_fepo_dr_grpo_group_advantages(all_correct)
    assert all(item["grpo_advantage"] == 0.0 for item in all_correct)
    assert all(item["failure_advantage"] == 0.0 for item in all_correct)
    assert all(item["failed_token_mask"].sum().item() == 0 for item in all_correct)
    print("FEPO Dr.GRPO advantage tests passed.")


run_reward_unit_tests()
run_fepo_dr_grpo_advantage_unit_tests()


## 7. Token-level FEPO losses and LoRA save/reload helpers

The clipped solver and failure losses use Dr.GRPO-style advantages from group mean-centering only. Group reward standard deviations are logged for diagnostics and are not part of the loss scale.


In [ ]:
def save_named_lora_adapter(adapter_name: str, adapter_dir: Path) -> Path:
    adapter_dir = Path(adapter_dir)
    if adapter_dir.exists():
        shutil.rmtree(adapter_dir)
    adapter_dir.mkdir(parents=True, exist_ok=True)
    set_active_policy_adapter(adapter_name, train=False)
    try:
        policy_model.save_pretrained(adapter_dir, selected_adapters=[adapter_name])
    except TypeError:
        policy_model.save_pretrained(adapter_dir)

    # PEFT saves non-default named adapters under adapter_dir/adapter_name.
    # vLLM's dynamic LoRA loader expects adapter_config.json at the loaded path,
    # so mirror the selected adapter files to adapter_dir itself.
    nested_adapter_dir = adapter_dir / adapter_name
    if nested_adapter_dir.exists() and (nested_adapter_dir / "adapter_config.json").exists():
        for child in nested_adapter_dir.iterdir():
            destination = adapter_dir / child.name
            if child.is_dir():
                if destination.exists():
                    shutil.rmtree(destination)
                shutil.copytree(child, destination)
            else:
                shutil.copy2(child, destination)

    if not (adapter_dir / "adapter_config.json").exists():
        raise FileNotFoundError(
            f"Saved adapter {adapter_name!r} is not vLLM-compatible at {adapter_dir}; "
            "adapter_config.json was not found."
        )

    tokenizer.save_pretrained(adapter_dir)
    return adapter_dir


def save_lora_adapter(step_label: Any) -> Path:
    adapter_dir = ADAPTER_ROOT / f"adapter_step_{step_label}"
    return save_named_lora_adapter(CONFIG.solver_adapter_name, adapter_dir)


def save_failure_lora_adapter(step_label: Any) -> Path:
    adapter_dir = FAILURE_ADAPTER_ROOT / f"adapter_step_{step_label}"
    return save_named_lora_adapter(CONFIG.failure_adapter_name, adapter_dir)


def cleanup_adapter_root(root: Path, current_adapter: Path) -> None:
    adapters = sorted(
        [p for p in Path(root).glob("adapter_step_*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    for stale in adapters[CONFIG.keep_last_adapters:]:
        if stale.resolve() != current_adapter.resolve():
            shutil.rmtree(stale, ignore_errors=True)


def cleanup_old_adapters(current_adapter: Path, current_failure_adapter: Optional[Path] = None) -> None:
    cleanup_adapter_root(ADAPTER_ROOT, current_adapter)
    if current_failure_adapter is not None:
        cleanup_adapter_root(FAILURE_ADAPTER_ROOT, current_failure_adapter)


def reload_vllm_lora(adapter_dir: Path) -> None:
    unload_lora_adapter(CONFIG.vllm_lora_name, ignore_missing=True)
    load_lora_adapter(CONFIG.vllm_lora_name, adapter_dir)


def save_reload_and_cleanup(step_label: Any) -> Tuple[Path, Path]:
    solver_adapter_dir = save_lora_adapter(step_label)
    failure_adapter_dir = save_failure_lora_adapter(step_label)
    reload_vllm_lora(solver_adapter_dir)
    cleanup_old_adapters(solver_adapter_dir, failure_adapter_dir)
    return solver_adapter_dir, failure_adapter_dir


def _zero_token_fepo_metrics(prefix: str = "solver", reference_beta: float = 0.0, escape_alpha: float = 0.0) -> Dict[str, float]:
    return {
        f"{prefix}_pg_loss": 0.0,
        f"{prefix}_pg_loss_abs": 0.0,
        f"{prefix}_total_loss": 0.0,
        f"{prefix}_clip_fraction": 0.0,
        "failure_escape_kl_failed": 0.0,
        "reference_kl_failed": 0.0,
        "failed_token_count": 0.0,
        "escape_alpha": float(escape_alpha),
        "reference_beta": float(reference_beta),
        "token_ratio_mean": 0.0,
        "token_log_ratio_abs_mean": 0.0,
        "loss_token_count": 0.0,
    }


def token_solver_fepo_loss(
    current_logps: torch.Tensor,
    old_logps: torch.Tensor,
    ref_logps: torch.Tensor,
    failure_logps: torch.Tensor,
    advantages: torch.Tensor,
    failed_token_mask: torch.Tensor,
    clip_eps: float,
    reference_beta: float,
    escape_alpha: float,
    loss_mask: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    if current_logps.numel() == 0:
        zero = current_logps.sum() * 0.0
        return zero, _zero_token_fepo_metrics("solver", reference_beta, escape_alpha)
    old_logps = old_logps.to(device=current_logps.device, dtype=current_logps.dtype).detach()
    ref_logps = ref_logps.to(device=current_logps.device, dtype=current_logps.dtype).detach()
    failure_logps = failure_logps.to(device=current_logps.device, dtype=current_logps.dtype).detach()
    advantages = advantages.to(device=current_logps.device, dtype=current_logps.dtype).detach()
    failed_token_mask = failed_token_mask.to(device=current_logps.device).bool().detach()
    n = min(current_logps.numel(), old_logps.numel(), ref_logps.numel(), failure_logps.numel(), advantages.numel(), failed_token_mask.numel())
    if loss_mask is not None:
        n = min(n, loss_mask.numel())
    current_logps = current_logps[:n]
    old_logps = old_logps[:n]
    ref_logps = ref_logps[:n]
    failure_logps = failure_logps[:n]
    advantages = advantages[:n]
    failed_token_mask = failed_token_mask[:n]
    if loss_mask is None:
        mask = torch.ones((n,), dtype=torch.bool, device=current_logps.device)
    else:
        mask = loss_mask[:n].to(device=current_logps.device).bool().detach()
    valid_count = int(mask.sum().detach().cpu())
    if valid_count == 0:
        zero = current_logps.sum() * 0.0
        return zero, _zero_token_fepo_metrics("solver", reference_beta, escape_alpha)

    current_logps = current_logps[mask]
    old_logps = old_logps[mask]
    ref_logps = ref_logps[mask]
    failure_logps = failure_logps[mask]
    advantages = advantages[mask]
    failed_token_mask = failed_token_mask[mask]

    log_ratio = current_logps - old_logps
    if CONFIG.token_log_ratio_clip is not None:
        log_ratio = log_ratio.clamp(-CONFIG.token_log_ratio_clip, CONFIG.token_log_ratio_clip)
    ratio = torch.exp(log_ratio)
    unclipped = ratio * advantages
    clipped_ratio = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps)
    clipped = clipped_ratio * advantages
    pg_terms = torch.minimum(unclipped, clipped)
    pg_loss = -pg_terms.mean()
    pg_loss_abs = pg_terms.abs().mean()
    clip_fraction = ((ratio < (1.0 - clip_eps)) | (ratio > (1.0 + clip_eps))).float().mean()

    failed_count = int(failed_token_mask.sum().detach().cpu())
    if failed_count > 0:
        failed_current_logps = current_logps[failed_token_mask]
        failed_failure_logps = failure_logps[failed_token_mask]
        failed_ref_logps = ref_logps[failed_token_mask]

        failure_minus_current = failed_failure_logps - failed_current_logps
        failure_escape_kl_failed = (
            torch.exp(failure_minus_current)
            - failure_minus_current
            - 1.0
        ).mean()

        ref_minus_current_failed = failed_ref_logps - failed_current_logps
        reference_kl_failed = (
            torch.exp(ref_minus_current_failed)
            - ref_minus_current_failed
            - 1.0
        ).mean()
    else:
        failure_escape_kl_failed = current_logps.sum() * 0.0
        reference_kl_failed = current_logps.sum() * 0.0

    loss = (
        pg_loss
        + float(reference_beta) * reference_kl_failed
        - float(escape_alpha) * failure_escape_kl_failed
    )
    metrics = {
        "solver_pg_loss": float(pg_loss.detach().cpu()),
        "solver_pg_loss_abs": float(pg_loss_abs.detach().cpu()),
        "failure_escape_kl_failed": float(failure_escape_kl_failed.detach().cpu()),
        "reference_kl_failed": float(reference_kl_failed.detach().cpu()),
        "failed_token_count": float(failed_count),
        "escape_alpha": float(escape_alpha),
        "reference_beta": float(reference_beta),
        "solver_total_loss": float(loss.detach().cpu()),
        "solver_clip_fraction": float(clip_fraction.detach().cpu()),
        "token_ratio_mean": float(ratio.detach().mean().cpu()),
        "token_log_ratio_abs_mean": float(log_ratio.detach().abs().mean().cpu()),
        "loss_token_count": float(valid_count),
    }
    return loss, metrics


def token_failure_grpo_loss(
    current_logps: torch.Tensor,
    old_logps: torch.Tensor,
    ref_logps: torch.Tensor,
    advantages: torch.Tensor,
    clip_eps: float,
    kl_beta: float,
    loss_mask: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    if current_logps.numel() == 0:
        zero = current_logps.sum() * 0.0
        return zero, {
            "failure_grpo_loss": 0.0,
            "failure_total_loss": 0.0,
            "failure_reference_point_kl": 0.0,
            "failure_token_ratio_mean": 0.0,
            "failure_token_log_ratio_abs_mean": 0.0,
            "failure_loss_token_count": 0.0,
        }
    old_logps = old_logps.to(device=current_logps.device, dtype=current_logps.dtype).detach()
    ref_logps = ref_logps.to(device=current_logps.device, dtype=current_logps.dtype).detach()
    advantages = advantages.to(device=current_logps.device, dtype=current_logps.dtype).detach()
    n = min(current_logps.numel(), old_logps.numel(), ref_logps.numel(), advantages.numel())
    if loss_mask is not None:
        n = min(n, loss_mask.numel())
    current_logps = current_logps[:n]
    old_logps = old_logps[:n]
    ref_logps = ref_logps[:n]
    advantages = advantages[:n]
    if loss_mask is None:
        mask = torch.ones((n,), dtype=torch.bool, device=current_logps.device)
    else:
        mask = loss_mask[:n].to(device=current_logps.device).bool().detach()
    valid_count = int(mask.sum().detach().cpu())
    if valid_count == 0:
        zero = current_logps.sum() * 0.0
        return zero, {
            "failure_grpo_loss": 0.0,
            "failure_total_loss": 0.0,
            "failure_reference_point_kl": 0.0,
            "failure_token_ratio_mean": 0.0,
            "failure_token_log_ratio_abs_mean": 0.0,
            "failure_loss_token_count": 0.0,
        }

    current_logps = current_logps[mask]
    old_logps = old_logps[mask]
    ref_logps = ref_logps[mask]
    advantages = advantages[mask]

    log_ratio = current_logps - old_logps
    if CONFIG.token_log_ratio_clip is not None:
        log_ratio = log_ratio.clamp(-CONFIG.token_log_ratio_clip, CONFIG.token_log_ratio_clip)
    ratio = torch.exp(log_ratio)
    unclipped = ratio * advantages
    clipped_ratio = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps)
    clipped = clipped_ratio * advantages
    grpo_terms = torch.minimum(unclipped, clipped)
    grpo_loss = -grpo_terms.mean()
    grpo_loss_abs = grpo_terms.abs().mean()
    clip_fraction = ((ratio < (1.0 - clip_eps)) | (ratio > (1.0 + clip_eps))).float().mean()

    ref_minus_current = ref_logps - current_logps
    reference_point_kl = (torch.exp(ref_minus_current) - ref_minus_current - 1.0).mean()
    loss = grpo_loss + float(kl_beta) * reference_point_kl
    metrics = {
        "failure_grpo_loss": float(grpo_loss.detach().cpu()),
        "failure_grpo_loss_abs": float(grpo_loss_abs.detach().cpu()),
        "failure_reference_point_kl": float(reference_point_kl.detach().cpu()),
        "failure_total_loss": float(loss.detach().cpu()),
        "failure_clip_fraction": float(clip_fraction.detach().cpu()),
        "failure_token_ratio_mean": float(ratio.detach().mean().cpu()),
        "failure_token_log_ratio_abs_mean": float(log_ratio.detach().abs().mean().cpu()),
        "failure_loss_token_count": float(valid_count),
    }
    return loss, metrics


def _zero_failure_sft_metrics() -> Dict[str, float]:
    return {
        "failure_sft_loss": 0.0,
        "failure_total_loss": 0.0,
        "failure_sft_token_count": 0.0,
        "failure_is_sft_active": 0.0,
    }


def token_failure_sft_loss(
    current_logps: torch.Tensor,
    is_failed: bool,
    loss_mask: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    if current_logps.numel() == 0 or not bool(is_failed):
        zero = current_logps.sum() * 0.0
        return zero, _zero_failure_sft_metrics()

    n = current_logps.numel()
    if loss_mask is not None:
        n = min(n, loss_mask.numel())
    current_logps = current_logps[:n]
    if loss_mask is None:
        mask = torch.ones((n,), dtype=torch.bool, device=current_logps.device)
    else:
        mask = loss_mask[:n].to(device=current_logps.device).bool().detach()

    valid_count = int(mask.sum().detach().cpu())
    if valid_count == 0:
        zero = current_logps.sum() * 0.0
        return zero, _zero_failure_sft_metrics()

    failure_sft_loss = -current_logps[mask].mean()
    metrics = {
        "failure_sft_loss": float(failure_sft_loss.detach().cpu()),
        "failure_total_loss": float(failure_sft_loss.detach().cpu()),
        "failure_sft_token_count": float(valid_count),
        "failure_is_sft_active": 1.0,
    }
    return failure_sft_loss, metrics


def detached_adapter_completion_logprobs(adapter_name: str, encoded: Dict[str, Any]) -> torch.Tensor:
    set_active_policy_adapter(adapter_name, train=False)
    return detached_completion_logprobs(policy_model, encoded)


def backward_one_response_solver_fepo(
    encoded: Dict[str, Any],
    old_solver_logps_cpu: torch.Tensor,
    ref_logps_cpu: torch.Tensor,
    failure_logps_cpu: torch.Tensor,
    token_advantages_cpu: torch.Tensor,
    failed_token_mask_cpu: torch.Tensor,
    token_loss_mask_cpu: Optional[torch.Tensor],
    scale: float,
) -> Dict[str, float]:
    set_active_policy_adapter(CONFIG.solver_adapter_name, train=True)
    current_logps = completion_logprobs(policy_model, encoded["input_ids"], encoded["completion_start"])
    loss, metrics = token_solver_fepo_loss(
        current_logps=current_logps,
        old_logps=old_solver_logps_cpu,
        ref_logps=ref_logps_cpu,
        failure_logps=failure_logps_cpu,
        advantages=token_advantages_cpu,
        failed_token_mask=failed_token_mask_cpu,
        clip_eps=CONFIG.clip_eps,
        reference_beta=CONFIG.fepo_reference_beta,
        escape_alpha=CONFIG.fepo_escape_alpha,
        loss_mask=token_loss_mask_cpu,
    )
    scaled_loss = loss * scale
    scaled_loss.backward()
    metrics["solver_scaled_loss"] = float(scaled_loss.detach().cpu())
    metrics["response_length"] = int(encoded["completion_length"])
    valid_adv = token_advantages_cpu
    if token_loss_mask_cpu is not None:
        mask = token_loss_mask_cpu[:token_advantages_cpu.numel()].bool()
        valid_adv = token_advantages_cpu[:mask.numel()][mask]
    metrics["advantage_mean"] = float(valid_adv.mean().item()) if valid_adv.numel() else 0.0
    metrics["advantage_std"] = float(valid_adv.std(unbiased=False).item()) if valid_adv.numel() else 0.0
    return metrics


def backward_one_response_failure_grpo(
    encoded: Dict[str, Any],
    old_failure_logps_cpu: torch.Tensor,
    ref_logps_cpu: torch.Tensor,
    failure_token_advantages_cpu: torch.Tensor,
    token_loss_mask_cpu: Optional[torch.Tensor],
    scale: float,
) -> Dict[str, float]:
    set_active_policy_adapter(CONFIG.failure_adapter_name, train=True)
    current_logps = completion_logprobs(policy_model, encoded["input_ids"], encoded["completion_start"])
    loss, metrics = token_failure_grpo_loss(
        current_logps=current_logps,
        old_logps=old_failure_logps_cpu,
        ref_logps=ref_logps_cpu,
        advantages=failure_token_advantages_cpu,
        clip_eps=CONFIG.failure_clip_eps,
        kl_beta=CONFIG.failure_kl_beta,
        loss_mask=token_loss_mask_cpu,
    )
    scaled_loss = loss * scale
    scaled_loss.backward()
    metrics["failure_scaled_loss"] = float(scaled_loss.detach().cpu())
    return metrics


def backward_one_response_failure_sft(
    encoded: Dict[str, Any],
    is_failed: bool,
    token_loss_mask_cpu: Optional[torch.Tensor],
    scale: float,
) -> Dict[str, float]:
    set_active_policy_adapter(CONFIG.failure_adapter_name, train=True)
    current_logps = completion_logprobs(policy_model, encoded["input_ids"], encoded["completion_start"])
    loss, metrics = token_failure_sft_loss(
        current_logps=current_logps,
        is_failed=is_failed,
        loss_mask=token_loss_mask_cpu,
    )
    scaled_loss = loss * scale
    scaled_loss.backward()
    metrics["failure_scaled_loss"] = float(scaled_loss.detach().cpu())
    return metrics


def run_token_fepo_unit_tests() -> None:
    current = torch.zeros(4, dtype=torch.float32, requires_grad=True)
    old = torch.zeros(4, dtype=torch.float32)
    ref = torch.tensor([0.0, 0.25, -0.25, 0.0], dtype=torch.float32)
    failure = torch.tensor([0.0, -0.5, 0.3, 0.0], dtype=torch.float32)
    adv = torch.tensor([0.5, -0.5, 0.25, 0.0], dtype=torch.float32)
    failed = torch.tensor([False, True, True, False])
    mask = torch.tensor([True, True, True, False])
    loss, metrics = token_solver_fepo_loss(
        current, old, ref, failure, adv, failed,
        CONFIG.clip_eps, CONFIG.fepo_reference_beta, CONFIG.fepo_escape_alpha, mask,
    )
    valid = mask.bool()
    failed_valid = failed.bool() & valid
    failure_minus_current = failure[failed_valid] - current.detach()[failed_valid]
    expected_failure_kl = (torch.exp(failure_minus_current) - failure_minus_current - 1.0).mean()
    ref_minus_current = ref[failed_valid] - current.detach()[failed_valid]
    expected_reference_kl = (torch.exp(ref_minus_current) - ref_minus_current - 1.0).mean()
    log_ratio = current.detach()[valid] - old[valid]
    expected_ratio = torch.exp(log_ratio)
    expected_pg_terms = torch.minimum(
        expected_ratio * adv[valid],
        torch.clamp(expected_ratio, 1.0 - CONFIG.clip_eps, 1.0 + CONFIG.clip_eps) * adv[valid],
    )
    expected_pg_loss = -expected_pg_terms.mean()
    expected_loss = (
        expected_pg_loss
        + CONFIG.fepo_reference_beta * expected_reference_kl
        - CONFIG.fepo_escape_alpha * expected_failure_kl
    )
    assert metrics["loss_token_count"] == 3.0
    assert metrics["failed_token_count"] == 2.0
    assert abs(metrics["failure_escape_kl_failed"] - float(expected_failure_kl.item())) < 1e-6
    assert abs(metrics["reference_kl_failed"] - float(expected_reference_kl.item())) < 1e-6
    assert abs(metrics["solver_pg_loss"] - float(expected_pg_loss.item())) < 1e-6
    assert torch.allclose(loss.detach(), expected_loss, atol=1e-6)
    assert torch.isfinite(loss)

    loss.backward()
    assert current.grad is not None
    assert torch.isfinite(current.grad).all()

    kl_current = torch.zeros(4, dtype=torch.float32, requires_grad=True)
    zero_adv = torch.zeros(4, dtype=torch.float32)
    kl_loss, kl_metrics = token_solver_fepo_loss(
        kl_current, old, ref, failure, zero_adv, failed,
        CONFIG.clip_eps, CONFIG.fepo_reference_beta, CONFIG.fepo_escape_alpha, mask,
    )
    kl_loss.backward()
    expected_failed_grad = (
        CONFIG.fepo_reference_beta * (1.0 - torch.exp(ref[failed_valid] - kl_current.detach()[failed_valid]))
        - CONFIG.fepo_escape_alpha * (1.0 - torch.exp(failure[failed_valid] - kl_current.detach()[failed_valid]))
    ) / float(failed_valid.sum().item())
    assert torch.allclose(kl_current.grad[failed_valid], expected_failed_grad, atol=1e-6)
    assert torch.allclose(kl_current.grad[valid & ~failed], torch.zeros_like(kl_current.grad[valid & ~failed]), atol=1e-6)
    assert kl_metrics["failed_token_count"] == 2.0

    no_failed_current = torch.zeros(4, dtype=torch.float32, requires_grad=True)
    no_failed = torch.zeros(4, dtype=torch.bool)
    no_failed_loss, no_failed_metrics = token_solver_fepo_loss(
        no_failed_current, old, ref, failure, zero_adv, no_failed,
        CONFIG.clip_eps, CONFIG.fepo_reference_beta, CONFIG.fepo_escape_alpha, mask,
    )
    assert no_failed_metrics["failed_token_count"] == 0.0
    assert no_failed_metrics["failure_escape_kl_failed"] == 0.0
    assert no_failed_metrics["reference_kl_failed"] == 0.0
    assert torch.isfinite(no_failed_loss)
    no_failed_loss.backward()
    assert no_failed_current.grad is not None
    assert torch.isfinite(no_failed_current.grad).all()

    sft_current = torch.tensor([-1.0, -2.0, -3.0, -4.0], dtype=torch.float32, requires_grad=True)
    sft_mask = torch.tensor([True, False, True, False])
    sft_loss, sft_metrics = token_failure_sft_loss(sft_current, is_failed=True, loss_mask=sft_mask)
    expected_sft_loss = -sft_current[sft_mask].mean()
    assert sft_metrics["failure_sft_token_count"] == 2.0
    assert sft_metrics["failure_is_sft_active"] == 1.0
    assert abs(sft_metrics["failure_sft_loss"] - float(expected_sft_loss.detach().item())) < 1e-6
    assert torch.allclose(sft_loss, expected_sft_loss, atol=1e-6)
    sft_loss.backward()
    expected_sft_grad = torch.tensor([-0.5, 0.0, -0.5, 0.0], dtype=torch.float32)
    assert torch.allclose(sft_current.grad, expected_sft_grad, atol=1e-6)

    correct_current = torch.tensor([-0.5, -1.5, -2.5], dtype=torch.float32, requires_grad=True)
    correct_loss, correct_metrics = token_failure_sft_loss(correct_current, is_failed=False, loss_mask=torch.ones(3, dtype=torch.bool))
    assert correct_metrics["failure_sft_loss"] == 0.0
    assert correct_metrics["failure_total_loss"] == 0.0
    assert correct_metrics["failure_sft_token_count"] == 0.0
    assert correct_metrics["failure_is_sft_active"] == 0.0
    assert torch.isfinite(correct_loss)
    correct_loss.backward()
    assert correct_current.grad is not None
    assert torch.allclose(correct_current.grad, torch.zeros_like(correct_current), atol=1e-6)

    short_wrong = torch.tensor([-2.0], dtype=torch.float32, requires_grad=True)
    long_wrong = torch.tensor([-2.0, -2.0, -2.0, -2.0], dtype=torch.float32, requires_grad=True)
    short_loss, short_metrics = token_failure_sft_loss(short_wrong, is_failed=True, loss_mask=torch.ones(1, dtype=torch.bool))
    long_loss, long_metrics = token_failure_sft_loss(long_wrong, is_failed=True, loss_mask=torch.ones(4, dtype=torch.bool))
    assert short_metrics["failure_sft_token_count"] == 1.0
    assert long_metrics["failure_sft_token_count"] == 4.0
    assert torch.allclose(short_loss, long_loss, atol=1e-6)
    assert torch.isfinite(sft_loss)
    print("Token FEPO/SFT point-loss tests passed.")


run_token_fepo_unit_tests()


def cuda_memory_summary_dict() -> Dict[str, Any]:
    if not torch.cuda.is_available():
        return {}
    return {
        "allocated_gb": round(torch.cuda.memory_allocated(TRAIN_DEVICE) / 1e9, 3),
        "reserved_gb": round(torch.cuda.memory_reserved(TRAIN_DEVICE) / 1e9, 3),
        "max_allocated_gb": round(torch.cuda.max_memory_allocated(TRAIN_DEVICE) / 1e9, 3),
    }


## 8. Launch vLLM and load the initial LoRA adapter

This cell launches the server, unloads any stale adapter name, loads the active adapter, checks `/v1/models`, and generates one response.


In [ ]:
launch_vllm()
if RESUME_CHECKPOINT_PATH is not None and checkpoint_adapter_path(RESUME_CHECKPOINT_PATH).exists():
    current_adapter_path = checkpoint_adapter_path(RESUME_CHECKPOINT_PATH)
    reload_vllm_lora(current_adapter_path)
    print("Loaded checkpoint LoRA adapter into vLLM:", current_adapter_path)
else:
    current_adapter_path, current_failure_adapter_path = save_reload_and_cleanup("initial")
models_payload = get_vllm_models()
print(json.dumps(models_payload, indent=2)[:2000])

smoke_generations = generate_with_vllm(
    train_examples[0]["messages"],
    n=1,
    temperature=0.0,
    top_p=1.0,
    max_tokens=64,
    timeout_s=CONFIG.request_timeout_s,
)
assert isinstance(smoke_generations[0], str)
print("vLLM LoRA generation smoke response:")
print(smoke_generations[0][:1000])


## 9. Rollout, logging, AMC23 evaluation, and FEPO training functions

The training loop processes one DAPO-Math-17K prompt at a time. Evaluation processes AMC23 prompts in batches and generates `CONFIG.eval_generations` answers per prompt.


In [ ]:
def write_jsonl(record: Dict[str, Any], path: Path = LOG_PATH) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


_wandb_disabled_after_error = False


_WANDB_METRIC_MAPS: Dict[str, Dict[str, str]] = {
    "train/response": {
        "solver_total_loss": "train/response/solver_total_loss",
        "solver_scaled_loss": "train/response/solver_scaled_loss",
        "solver_pg_loss": "train/response/solver_pg_loss",
        "solver_pg_loss_abs": "train/response/solver_pg_loss_abs",
        "failure_escape_kl_failed": "fepo/failure_escape_kl_failed",
        "reference_kl_failed": "fepo/reference_kl_failed",
        "escape_alpha": "fepo/escape_alpha",
        "reference_beta": "fepo/reference_beta",
        "solver_clip_fraction": "grpo/solver_clip_fraction",
        "failure_sft_loss": "train/response/failure_sft_loss",
        "failure_total_loss": "train/response/failure_total_loss",
        "failure_scaled_loss": "train/response/failure_scaled_loss",
        "failure_sft_token_count": "tokens/failure_sft_token_count",
        "failure_is_sft_active": "failure/is_sft_active",
        "failure_active_response_count": "failure/active_response_count",
        "token_ratio_mean": "grpo/token_ratio_mean",
        "token_log_ratio_abs_mean": "grpo/token_log_ratio_abs_mean",
        "advantage_mean": "train/response/token_advantage_mean",
        "advantage_std": "train/response/token_advantage_std",
        "loss_token_count": "tokens/loss_token_count",
        "valid_token_count": "tokens/valid_count",
        "ignored_token_count": "tokens/ignored_count",
        "response_length": "tokens/response_length",
        "reward": "reward/value",
        "is_correct": "reward/is_correct",
        "has_parseable_answer": "reward/parseable",
        "grpo_advantage": "grpo/advantage",
        "anti_reward": "failure/anti_reward",
        "failure_advantage": "failure/advantage",
        "group_reward_mean": "reward/group_mean",
        "group_reward_std": "reward/group_std",
        "group_anti_reward_mean": "failure/group_anti_reward_mean",
        "group_anti_reward_std": "failure/group_anti_reward_std",
        "failed_token_count": "fepo/failed_token_count",
        "train_prompt_batch_index": "train/response/prompt_batch_index",
        "train_prompt_batch_size": "train/response/prompt_batch_size",
        "batch_accuracy": "reward/batch_accuracy",
        "batch_parse_rate": "reward/batch_parse_rate",
    },
    "train/optimizer": {
        "solver_total_loss_mean": "train/optimizer/solver_total_loss_mean",
        "solver_total_loss_sum": "train/optimizer/solver_total_loss_sum",
        "solver_pg_loss_mean": "train/optimizer/solver_pg_loss_mean",
        "solver_pg_loss_abs_mean": "train/optimizer/solver_pg_loss_abs_mean",
        "failure_escape_kl_failed_mean": "fepo/failure_escape_kl_failed_mean",
        "reference_kl_failed_mean": "fepo/reference_kl_failed_mean",
        "escape_alpha_mean": "fepo/escape_alpha_mean",
        "reference_beta_mean": "fepo/reference_beta_mean",
        "solver_clip_fraction_mean": "grpo/solver_clip_fraction_mean",
        "failure_sft_loss_mean": "train/optimizer/failure_sft_loss_mean",
        "failure_total_loss_mean": "train/optimizer/failure_total_loss_mean",
        "failure_sft_token_count_sum": "tokens/failure_sft_token_count_sum",
        "failure_is_sft_active_mean": "failure/is_sft_active_mean",
        "failure_active_response_count_mean": "failure/active_response_count_mean",
        "token_ratio_mean": "grpo/token_ratio_mean_step",
        "token_log_ratio_abs_mean": "grpo/token_log_ratio_abs_mean_step",
        "train_accuracy": "train/accuracy",
        "train_failure_rate": "train/failure_rate",
        "train_parse_rate": "train/parse_rate",
        "train_correct_count": "train/correct_count",
        "train_total_count": "train/total_count",
        "reward_mean": "reward/mean",
        "reward_std": "reward/std",
        "reward_min": "reward/min",
        "reward_max": "reward/max",
        "parse_rate_mean": "reward/parse_rate_mean",
        "grpo_advantage_mean": "grpo/advantage_mean",
        "grpo_advantage_std": "grpo/advantage_std",
        "active_response_count_mean": "train/optimizer/active_response_count_mean",
        "train_prompt_batch_size_mean": "train/optimizer/prompt_batch_size_mean",
        "loss_token_count_sum": "tokens/loss_token_count_sum",
        "valid_token_count_sum": "tokens/valid_count_sum",
        "ignored_token_count_sum": "tokens/ignored_count_sum",
        "solver_grad_norm": "train/optimizer/solver_grad_norm",
        "failure_grad_norm": "train/optimizer/failure_grad_norm",
        "optimizer_step": "train/optimizer/step",
        "checkpoint_saved": "checkpoint/saved",
        "resumed_from_checkpoint": "checkpoint/resumed",
        "resume_optimizer_step": "checkpoint/resume_optimizer_step",
        "resume_completed_accumulations": "checkpoint/resume_completed_accumulations",
        "examples_seen": "checkpoint/examples_seen",
    },
    "eval": {
        "pass_at_1": "eval/pass_at_1",
        "pass_at_k": "eval/pass_at_k",
        "avg_at_k": "eval/avg_at_k",
        "parse_rate": "eval/parse_rate",
        "total_prompts": "eval/total_prompts",
        "total_generations": "eval/total_generations",
        "k": "eval/k",
        "step": "eval/step",
    },
}


def _wandb_get_nested(record: Dict[str, Any], dotted_key: str) -> Any:
    value: Any = record
    for part in dotted_key.split("."):
        if not isinstance(value, dict) or part not in value:
            return None
        value = value[part]
    return value


def _wandb_metric_payload(record: Dict[str, Any], prefix: str) -> Dict[str, Any]:
    mapping = _WANDB_METRIC_MAPS.get(prefix, {})
    payload: Dict[str, Any] = {}
    for source_key, target_key in mapping.items():
        value = _wandb_get_nested(record, source_key)
        if value is None:
            continue
        if isinstance(value, (bool, np.bool_)):
            payload[target_key] = int(value)
        elif isinstance(value, (int, float, np.integer, np.floating)) and math.isfinite(float(value)):
            payload[target_key] = float(value)
    return payload


def wandb_log_scalars(record: Dict[str, Any], prefix: str, optimizer_step: Optional[int] = None) -> None:
    global _wandb_disabled_after_error
    if WANDB_RUN is None or _wandb_disabled_after_error:
        return
    payload = _wandb_metric_payload(record, prefix)
    if not payload:
        return
    step = optimizer_step if optimizer_step is not None else record.get("optimizer_step")
    try:
        wandb.log(payload, step=step)
    except Exception as exc:
        _wandb_disabled_after_error = True
        print("W&B scalar logging failed; disabling further W&B logs:", exc)


def wandb_log_table(name: str, rows: List[Dict[str, Any]], optimizer_step: Optional[int] = None) -> None:
    global _wandb_disabled_after_error
    if WANDB_RUN is None or _wandb_disabled_after_error or not rows:
        return
    if wandb is None:
        return
    columns = list(rows[0].keys())
    table = wandb.Table(columns=columns)
    for row in rows:
        table.add_data(*[row.get(col) for col in columns])
    try:
        wandb.log({name: table}, step=optimizer_step)
    except Exception as exc:
        _wandb_disabled_after_error = True
        print("W&B table logging failed; disabling further W&B logs:", exc)


def truncate_text(text: Any, max_chars: int) -> str:
    text = "" if text is None else str(text)
    if len(text) <= max_chars:
        return text
    return text[: max(0, max_chars - 3)] + "..."


def _fmt_optional(record: Dict[str, Any], key: str, fmt: str = "{:.3f}") -> str:
    value = record.get(key)
    if value is None:
        return ""
    try:
        if not math.isfinite(float(value)):
            return str(value)
        return fmt.format(float(value))
    except Exception:
        return str(value)


def format_log_table(records: Sequence[Dict[str, Any]]) -> List[Dict[str, Any]]:
    table = []
    for record in records:
        table.append({
            "gen": record.get("generation_index"),
            "reward": _fmt_optional(record, "reward", "{:.1f}"),
            "anti": _fmt_optional(record, "anti_reward", "{:+.1f}"),
            "adv": _fmt_optional(record, "grpo_advantage", "{:+.3f}"),
            "fail_adv": _fmt_optional(record, "failure_advantage", "{:+.3f}"),
            "pred": str(record.get("prediction_normalized")),
            "target": str(record.get("ground_truth_normalized")),
            "correct": record.get("is_correct"),
            "parseable": record.get("has_parseable_answer"),
            "loss": _fmt_optional(record, "response_loss", "{:+.4f}"),
            "tokens": record.get("valid_token_count", 0),
            "response": truncate_text(record.get("response", ""), CONFIG.log_response_chars),
        })
    return table


def print_records_table(records: Sequence[Dict[str, Any]]) -> None:
    if not records:
        return
    df = pd.DataFrame(format_log_table(records))
    with pd.option_context("display.max_colwidth", CONFIG.log_response_chars, "display.width", 240):
        print(df.to_string(index=False))


def generate_train_responses(example: Dict[str, Any]) -> List[str]:
    outputs = generate_with_vllm(
        messages=example["messages"],
        n=CONFIG.num_generations,
        temperature=CONFIG.temperature,
        top_p=CONFIG.top_p,
        max_tokens=CONFIG.train_max_completion_tokens,
    )
    if len(outputs) != int(CONFIG.num_generations):
        raise RuntimeError(f"vLLM returned {len(outputs)} generations, expected {CONFIG.num_generations}.")
    return outputs


def build_rollout_items(example: Dict[str, Any], outputs: Sequence[str]) -> List[Dict[str, Any]]:
    items: List[Dict[str, Any]] = []
    for generation_index, response in enumerate(outputs):
        encoded = encode_prompt_response(example["messages"], response)
        if encoded is None:
            item = {
                "response": response,
                "generation_index": generation_index,
                "encoded": None,
                "old_solver_logps": torch.zeros((0,), dtype=torch.float32),
                "old_failure_logps": torch.zeros((0,), dtype=torch.float32),
                "ref_logps": torch.zeros((0,), dtype=torch.float32),
                "failure_logps_for_escape": torch.zeros((0,), dtype=torch.float32),
            }
        else:
            old_solver_logps = detached_adapter_completion_logprobs(CONFIG.solver_adapter_name, encoded)
            old_failure_logps = detached_adapter_completion_logprobs(CONFIG.failure_adapter_name, encoded)
            failure_logps_for_escape = old_failure_logps.clone()
            ref_stats = detached_reference_completion_stats(encoded)
            item = {
                "response": response,
                "generation_index": generation_index,
                "encoded": encoded,
                "old_solver_logps": old_solver_logps,
                "old_failure_logps": old_failure_logps,
                "ref_logps": ref_stats["ref_logps"],
                "failure_logps_for_escape": failure_logps_for_escape,
            }
        items.append(item)
        gc.collect()
        torch.cuda.empty_cache()

    compute_ground_truth_rewards(example, items)
    attach_fepo_dr_grpo_group_advantages(items)
    return items


def rollout_prompt(example: Dict[str, Any]) -> List[Dict[str, Any]]:
    set_active_policy_adapter(CONFIG.solver_adapter_name, train=False)
    reference_model.eval()
    return build_rollout_items(example, generate_train_responses(example))


def rollout_prompt_batch(examples: Sequence[Dict[str, Any]]) -> List[Tuple[Dict[str, Any], List[Dict[str, Any]]]]:
    examples = list(examples)
    if not examples:
        return []
    set_active_policy_adapter(CONFIG.solver_adapter_name, train=False)
    reference_model.eval()

    max_workers = max(1, min(len(examples), int(CONFIG.train_prompt_batch_size)))
    outputs_by_index: Dict[int, List[str]] = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {
            executor.submit(generate_train_responses, example): idx
            for idx, example in enumerate(examples)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            outputs_by_index[idx] = future.result()

    rollouts: List[Tuple[Dict[str, Any], List[Dict[str, Any]]]] = []
    for idx, example in enumerate(examples):
        rollouts.append((example, build_rollout_items(example, outputs_by_index[idx])))
    return rollouts


def active_response_count_for_items(items: Sequence[Dict[str, Any]]) -> int:
    active_response_count = 0
    for item in items:
        mask = item.get("token_loss_mask")
        if mask is not None and int(mask.sum().item()) > 0:
            active_response_count += 1
    return active_response_count


def failure_active_response_count_for_items(items: Sequence[Dict[str, Any]]) -> int:
    return sum(
        1
        for item in items
        if item.get("encoded") is not None
        and int(item.get("valid_token_count", 0)) > 0
        and float(item.get("reward", 0.0)) == 0.0
    )

def find_nonfinite_gradients(model: torch.nn.Module, max_items: int = 8, adapter_name: Optional[str] = None) -> List[Dict[str, Any]]:
    bad = []
    marker = _adapter_name_marker(adapter_name) if adapter_name is not None else None
    for name, param in model.named_parameters():
        if marker is not None and marker not in name:
            continue
        grad = param.grad
        if grad is None:
            continue
        finite_mask = torch.isfinite(grad)
        if bool(finite_mask.all()):
            continue
        detached = grad.detach()
        bad.append({
            "name": name,
            "shape": tuple(detached.shape),
            "nan_count": int(torch.isnan(detached).sum().cpu()),
            "inf_count": int(torch.isinf(detached).sum().cpu()),
            "finite_count": int(finite_mask.sum().cpu()),
        })
        if len(bad) >= max_items:
            break
    return bad


def assert_finite_gradients(model: torch.nn.Module, context: str, adapter_name: Optional[str] = None) -> None:
    bad = find_nonfinite_gradients(model, adapter_name=adapter_name)
    if not bad:
        return
    message = json.dumps(bad, indent=2)
    if CONFIG.stop_on_nonfinite_grad:
        raise FloatingPointError(f"Non-finite gradients after {context}:\n{message}")
    print(f"WARNING: non-finite gradients after {context}:\n{message}")


def _zero_encoded_training_metrics() -> Dict[str, float]:
    return {
        "solver_pg_loss": 0.0,
        "solver_pg_loss_abs": 0.0,
        "failure_escape_kl_failed": 0.0,
        "reference_kl_failed": 0.0,
        "escape_alpha": float(CONFIG.fepo_escape_alpha),
        "reference_beta": float(CONFIG.fepo_reference_beta),
        "solver_total_loss": 0.0,
        "solver_scaled_loss": 0.0,
        "solver_clip_fraction": 0.0,
        "failure_sft_loss": 0.0,
        "failure_total_loss": 0.0,
        "failure_scaled_loss": 0.0,
        "failure_sft_token_count": 0.0,
        "failure_is_sft_active": 0.0,
        "token_ratio_mean": 0.0,
        "token_log_ratio_abs_mean": 0.0,
        "response_length": 0,
        "advantage_mean": 0.0,
        "advantage_std": 0.0,
        "loss_token_count": 0.0,
        "failed_token_count": 0.0,
        "failure_active_response_count": 0.0,
    }


def train_on_prompt_items(
    example: Dict[str, Any],
    items: Sequence[Dict[str, Any]],
    optimizer_step: int,
    accumulation_index: int,
    adapter_path: Path,
    tag: str,
    response_scale: float,
    active_response_count: int,
    failure_response_scale: float,
    failure_active_response_count: int,
    train_prompt_batch_index: int = 0,
    train_prompt_batch_size: int = 1,
) -> List[Dict[str, Any]]:
    records = []

    for item in items:
        generation_index = int(item.get("generation_index", len(records)))
        encoded = item.get("encoded")
        if encoded is None:
            metrics = _zero_encoded_training_metrics()
        else:
            solver_metrics = backward_one_response_solver_fepo(
                encoded=encoded,
                old_solver_logps_cpu=item["old_solver_logps"],
                ref_logps_cpu=item["ref_logps"],
                failure_logps_cpu=item["failure_logps_for_escape"],
                token_advantages_cpu=item["token_advantages"],
                failed_token_mask_cpu=item["failed_token_mask"],
                token_loss_mask_cpu=item.get("token_loss_mask"),
                scale=response_scale,
            )
            assert_finite_gradients(
                policy_model,
                context=(
                    f"solver step={optimizer_step} accum={accumulation_index} "
                    f"prompt_batch={train_prompt_batch_index} gen={generation_index}"
                ),
                adapter_name=CONFIG.solver_adapter_name,
            )
            failure_metrics = backward_one_response_failure_sft(
                encoded=encoded,
                is_failed=(float(item.get("reward", 0.0)) == 0.0),
                token_loss_mask_cpu=item.get("token_loss_mask"),
                scale=failure_response_scale,
            )
            assert_finite_gradients(
                policy_model,
                context=(
                    f"failure step={optimizer_step} accum={accumulation_index} "
                    f"prompt_batch={train_prompt_batch_index} gen={generation_index}"
                ),
                adapter_name=CONFIG.failure_adapter_name,
            )
            metrics = {**solver_metrics, **failure_metrics}

        record = {
            "tag": tag,
            "algorithm": CONFIG.algorithm_name,
            "dataset": example.get("dataset"),
            "optimizer_step": optimizer_step,
            "accumulation_index": accumulation_index,
            "train_prompt_batch_index": int(train_prompt_batch_index),
            "train_prompt_batch_size": int(train_prompt_batch_size),
            "generation_index": generation_index,
            "adapter_path": str(adapter_path),
            "example_id": example.get("example_id"),
            "url": example.get("url"),
            "problem": example["problem"],
            "response": item["response"],
            "prediction_normalized": item.get("prediction_normalized"),
            "ground_truth_normalized": item.get("ground_truth_normalized"),
            "has_parseable_answer": bool(item.get("has_parseable_answer", False)),
            "reward": float(item.get("reward", 0.0)),
            "anti_reward": float(item.get("anti_reward", 1.0)),
            "is_correct": bool(item.get("is_correct", False)),
            "grpo_advantage": float(item.get("grpo_advantage", 0.0)),
            "failure_advantage": float(item.get("failure_advantage", 0.0)),
            "group_reward_mean": float(item.get("group_reward_mean", 0.0)),
            "group_reward_std": float(item.get("group_reward_std", 0.0)),
            "group_anti_reward_mean": float(item.get("group_anti_reward_mean", 0.0)),
            "group_anti_reward_std": float(item.get("group_anti_reward_std", 0.0)),
            "active_response_count": int(active_response_count),
            "failure_active_response_count": int(failure_active_response_count),
            "valid_token_count": int(item.get("valid_token_count", 0)),
            "ignored_token_count": int(item.get("ignored_token_count", 0)),
            **metrics,
            "cuda_memory": cuda_memory_summary_dict(),
        }
        records.append(record)
        gc.collect()
        torch.cuda.empty_cache()

    batch_total = len(records)
    batch_correct = sum(1 for r in records if r.get("is_correct") is True)
    batch_parseable = sum(1 for r in records if r.get("has_parseable_answer") is True)
    batch_accuracy = float(batch_correct / batch_total) if batch_total else 0.0
    batch_parse_rate = float(batch_parseable / batch_total) if batch_total else 0.0

    for record in records:
        record["batch_correct"] = int(batch_correct)
        record["batch_total"] = int(batch_total)
        record["batch_accuracy"] = float(batch_accuracy)
        record["batch_parse_rate"] = float(batch_parse_rate)
        write_jsonl(record)
        wandb_log_scalars(record, prefix="train/response", optimizer_step=optimizer_step)

    if CONFIG.print_table_every_prompt:
        print_records_table(records)
    if CONFIG.wandb_log_tables and records:
        train_table = format_log_table(records)
        wandb_log_table("train/prompt_records", train_table, optimizer_step=optimizer_step)
    return records


def train_on_prompt(
    example: Dict[str, Any],
    optimizer_step: int,
    accumulation_index: int,
    adapter_path: Path,
    tag: str,
) -> List[Dict[str, Any]]:
    items = rollout_prompt(example)
    active_response_count = max(active_response_count_for_items(items), 1)
    failure_active_response_count = max(failure_active_response_count_for_items(items), 1)
    response_scale = 1.0 / (CONFIG.gradient_accumulation_steps * active_response_count)
    failure_response_scale = 1.0 / (CONFIG.gradient_accumulation_steps * failure_active_response_count)
    return train_on_prompt_items(
        example=example,
        items=items,
        optimizer_step=optimizer_step,
        accumulation_index=accumulation_index,
        adapter_path=adapter_path,
        tag=tag,
        response_scale=response_scale,
        active_response_count=active_response_count,
        failure_response_scale=failure_response_scale,
        failure_active_response_count=failure_active_response_count,
        train_prompt_batch_index=0,
        train_prompt_batch_size=1,
    )


def train_on_prompt_batch(
    examples: Sequence[Dict[str, Any]],
    optimizer_step: int,
    accumulation_index: int,
    adapter_path: Path,
    tag: str,
) -> List[Dict[str, Any]]:
    rollouts = rollout_prompt_batch(examples)
    active_response_count = max(
        sum(active_response_count_for_items(items) for _, items in rollouts),
        1,
    )
    failure_active_response_count = max(
        sum(failure_active_response_count_for_items(items) for _, items in rollouts),
        1,
    )
    response_scale = 1.0 / (CONFIG.gradient_accumulation_steps * active_response_count)
    failure_response_scale = 1.0 / (
        CONFIG.gradient_accumulation_steps * failure_active_response_count
    )
    records: List[Dict[str, Any]] = []
    prompt_batch_size = len(rollouts)
    for prompt_batch_index, (example, items) in enumerate(rollouts):
        records.extend(train_on_prompt_items(
            example=example,
            items=items,
            optimizer_step=optimizer_step,
            accumulation_index=accumulation_index,
            adapter_path=adapter_path,
            tag=tag,
            response_scale=response_scale,
            active_response_count=active_response_count,
            failure_response_scale=failure_response_scale,
            failure_active_response_count=failure_active_response_count,
            train_prompt_batch_index=prompt_batch_index,
            train_prompt_batch_size=prompt_batch_size,
        ))
    return records

def _finite_array(values: Iterable[Any]) -> np.ndarray:
    finite = []
    for value in values:
        try:
            value = float(value)
        except Exception:
            continue
        if math.isfinite(value):
            finite.append(value)
    return np.asarray(finite, dtype=np.float64)


def summarize_step_records(records: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    if not records:
        return {}
    rewards = _finite_array(r.get("reward", 0.0) for r in records)
    train_total_count = int(len(records))
    train_correct_count = int(sum(1 for r in records if bool(r.get("is_correct", False))))
    train_parseable_count = int(sum(1 for r in records if bool(r.get("has_parseable_answer", False))))
    train_accuracy = float(train_correct_count / train_total_count) if train_total_count else 0.0
    train_parse_rate = float(train_parseable_count / train_total_count) if train_total_count else 0.0
    train_failure_rate = float(1.0 - train_accuracy) if train_total_count else 0.0
    advantages = _finite_array(r.get("grpo_advantage", 0.0) for r in records)
    failure_advantages = _finite_array(r.get("failure_advantage", 0.0) for r in records)
    solver_losses = _finite_array(r.get("solver_total_loss", 0.0) for r in records)
    solver_pg_losses = _finite_array(r.get("solver_pg_loss", 0.0) for r in records)
    solver_pg_loss_abs = _finite_array(r.get("solver_pg_loss_abs", 0.0) for r in records)
    failure_escape_kls_failed = _finite_array(r.get("failure_escape_kl_failed", 0.0) for r in records if float(r.get("failed_token_count", 0.0)) > 0.0)
    reference_kls_failed = _finite_array(r.get("reference_kl_failed", 0.0) for r in records if float(r.get("failed_token_count", 0.0)) > 0.0)
    escape_alphas = _finite_array(r.get("escape_alpha", 0.0) for r in records)
    reference_betas = _finite_array(r.get("reference_beta", 0.0) for r in records)
    solver_clip_fractions = _finite_array(r.get("solver_clip_fraction", 0.0) for r in records)
    active_failure_records = [r for r in records if float(r.get("failure_is_sft_active", 0.0)) > 0.0]
    failure_losses = _finite_array(r.get("failure_total_loss", 0.0) for r in active_failure_records)
    failure_sft_losses = _finite_array(r.get("failure_sft_loss", 0.0) for r in active_failure_records)
    failure_sft_active_flags = _finite_array(r.get("failure_is_sft_active", 0.0) for r in records)
    failure_active_counts = _finite_array(r.get("failure_active_response_count", 0.0) for r in records)
    ratios = _finite_array(r.get("token_ratio_mean", 0.0) for r in records)
    log_ratio_abs = _finite_array(r.get("token_log_ratio_abs_mean", 0.0) for r in records)
    active_counts = _finite_array(r.get("active_response_count", 0.0) for r in records)
    train_prompt_batch_sizes = _finite_array(r.get("train_prompt_batch_size", 1.0) for r in records)
    parse_rates = _finite_array(r.get("batch_parse_rate", 0.0) for r in records)

    return {
        "optimizer_step": int(records[-1].get("optimizer_step", 0)),
        "record_count": int(len(records)),
        "train_accuracy": float(train_accuracy),
        "train_failure_rate": float(train_failure_rate),
        "train_parse_rate": float(train_parse_rate),
        "train_correct_count": int(train_correct_count),
        "train_parseable_count": int(train_parseable_count),
        "train_total_count": int(train_total_count),
        "solver_total_loss_mean": float(solver_losses.mean()) if solver_losses.size else 0.0,
        "solver_total_loss_sum": float(solver_losses.sum()) if solver_losses.size else 0.0,
        "solver_pg_loss_mean": float(solver_pg_losses.mean()) if solver_pg_losses.size else 0.0,
        "solver_pg_loss_abs_mean": float(solver_pg_loss_abs.mean()) if solver_pg_loss_abs.size else 0.0,
        "failure_escape_kl_failed_mean": float(failure_escape_kls_failed.mean()) if failure_escape_kls_failed.size else 0.0,
        "reference_kl_failed_mean": float(reference_kls_failed.mean()) if reference_kls_failed.size else 0.0,
        "escape_alpha_mean": float(escape_alphas.mean()) if escape_alphas.size else 0.0,
        "reference_beta_mean": float(reference_betas.mean()) if reference_betas.size else 0.0,
        "solver_clip_fraction_mean": float(solver_clip_fractions.mean()) if solver_clip_fractions.size else 0.0,
        "failure_total_loss_mean": float(failure_losses.mean()) if failure_losses.size else 0.0,
        "failure_sft_loss_mean": float(failure_sft_losses.mean()) if failure_sft_losses.size else 0.0,
        "failure_sft_token_count_sum": float(sum(float(r.get("failure_sft_token_count", 0.0)) for r in records)),
        "failure_is_sft_active_mean": float(failure_sft_active_flags.mean()) if failure_sft_active_flags.size else 0.0,
        "failure_active_response_count_mean": float(failure_active_counts.mean()) if failure_active_counts.size else 0.0,
        "token_ratio_mean": float(ratios.mean()) if ratios.size else 0.0,
        "token_log_ratio_abs_mean": float(log_ratio_abs.mean()) if log_ratio_abs.size else 0.0,
        "reward_mean": float(rewards.mean()) if rewards.size else 0.0,
        "reward_std": float(rewards.std()) if rewards.size else 0.0,
        "reward_min": float(rewards.min()) if rewards.size else 0.0,
        "reward_max": float(rewards.max()) if rewards.size else 0.0,
        "parse_rate_mean": float(parse_rates.mean()) if parse_rates.size else 0.0,
        "grpo_advantage_mean": float(advantages.mean()) if advantages.size else 0.0,
        "grpo_advantage_std": float(advantages.std()) if advantages.size else 0.0,
        "failure_advantage_mean": float(failure_advantages.mean()) if failure_advantages.size else 0.0,
        "failure_advantage_std": float(failure_advantages.std()) if failure_advantages.size else 0.0,
        "active_response_count_mean": float(active_counts.mean()) if active_counts.size else 0.0,
        "train_prompt_batch_size_mean": float(train_prompt_batch_sizes.mean()) if train_prompt_batch_sizes.size else 1.0,
        "loss_token_count_sum": float(sum(float(r.get("loss_token_count", 0.0)) for r in records)),
        "failed_token_count_sum": float(sum(float(r.get("failed_token_count", 0.0)) for r in records)),
        "valid_token_count_sum": int(sum(int(r.get("valid_token_count", 0)) for r in records)),
        "ignored_token_count_sum": int(sum(int(r.get("ignored_token_count", 0)) for r in records)),
        "checkpoint_saved": False,
    }


def print_optimizer_step_summary(summary: Dict[str, Any], grad_norms: Dict[str, Optional[float]], adapter_path: Path) -> None:
    if not summary:
        return
    compact = {
        "step": summary.get("optimizer_step"),
        "solver_loss": f"{summary.get('solver_total_loss_mean', 0.0):+.6f}",
        "solver_abs": f"{summary.get('solver_pg_loss_abs_mean', 0.0):.6f}",
        "failure_loss": f"{summary.get('failure_total_loss_mean', 0.0):+.6f}",
        "failure_sft": f"{summary.get('failure_sft_loss_mean', 0.0):+.6f}",
        "train_acc": f"{summary.get('train_accuracy', 0.0):.3f}",
        "train_fail": f"{summary.get('train_failure_rate', 0.0):.3f}",
        "train_parse": f"{summary.get('train_parse_rate', 0.0):.3f}",
        "reward": (
            f"{summary.get('reward_mean', 0.0):.3f}/"
            f"{summary.get('reward_min', 0.0):.1f}/"
            f"{summary.get('reward_max', 0.0):.1f}"
        ),
        "solver_adv": f"{summary.get('grpo_advantage_mean', 0.0):+.3f}/{summary.get('grpo_advantage_std', 0.0):.3f}",
        "failure_adv": f"{summary.get('failure_advantage_mean', 0.0):+.3f}/{summary.get('failure_advantage_std', 0.0):.3f}",
        "escape_kl_failed": f"{summary.get('failure_escape_kl_failed_mean', 0.0):.6f}",
        "ref_kl_failed": f"{summary.get('reference_kl_failed_mean', 0.0):.6f}",
        "alpha_beta": f"{summary.get('escape_alpha_mean', 0.0):.4f}/{summary.get('reference_beta_mean', 0.0):.4f}",
        "solver_clip": f"{summary.get('solver_clip_fraction_mean', 0.0):.3f}",
        "failed_tokens": f"{summary.get('failed_token_count_sum', 0.0):.0f}",
        "failure_sft_tokens": f"{summary.get('failure_sft_token_count_sum', 0.0):.0f}",
        "failure_active": f"{summary.get('failure_is_sft_active_mean', 0.0):.3f}/{summary.get('failure_active_response_count_mean', 0.0):.1f}",
        "prompt_batch": f"{summary.get('train_prompt_batch_size_mean', 1.0):.1f}",
        "solver_grad": f"{grad_norms.get('solver'):.3f}" if grad_norms.get('solver') is not None else "None",
        "failure_grad": f"{grad_norms.get('failure'):.3f}" if grad_norms.get('failure') is not None else "None",
        "adapter": truncate_text(str(adapter_path), 50),
        "ckpt": truncate_text(summary.get("checkpoint_path", ""), 50),
    }
    print("Optimizer summary:", json.dumps(compact, ensure_ascii=False))


def finish_wandb_run(final_eval: Optional[Dict[str, Any]] = None) -> None:
    if WANDB_RUN is None:
        return
    try:
        WANDB_RUN.summary["final_adapter_dir"] = str(FINAL_ADAPTER_DIR)
        WANDB_RUN.summary["final_failure_adapter_dir"] = str(FINAL_FAILURE_ADAPTER_DIR)
        WANDB_RUN.summary["log_jsonl"] = str(LOG_PATH)
        WANDB_RUN.summary["vllm_log"] = str(VLLM_LOG_PATH)
        WANDB_RUN.summary["algorithm"] = CONFIG.algorithm_name
        if final_eval is not None:
            for key, value in final_eval.get("summary", {}).items():
                WANDB_RUN.summary[f"final_eval/{key}"] = value
        if CONFIG.wandb_log_model_artifact and FINAL_ADAPTER_DIR.exists():
            artifact = wandb.Artifact(
                name=f"{WANDB_RUN.name}-final-lora",
                type="model",
                metadata={
                    "model_id": CONFIG.model_id,
                    "algorithm": CONFIG.algorithm_name,
                    "adapter_dir": str(FINAL_ADAPTER_DIR),
                },
            )
            artifact.add_dir(str(FINAL_ADAPTER_DIR))
            WANDB_RUN.log_artifact(artifact)
        wandb.finish()
    except Exception as exc:
        print("W&B finish failed:", exc)


class StatefulExampleStream:
    def __init__(self, examples: Sequence[Dict[str, Any]], seed: int):
        self.examples = list(examples)
        self.seed = int(seed)
        self.rng = random.Random(self.seed)
        self.order: List[int] = []
        self.position = 0
        self.epoch = 0
        self.examples_seen = 0
        self._reshuffle()

    def _reshuffle(self) -> None:
        self.order = list(range(len(self.examples)))
        self.rng.shuffle(self.order)
        self.position = 0
        self.epoch += 1

    def __iter__(self):
        return self

    def __next__(self) -> Dict[str, Any]:
        if not self.examples:
            raise StopIteration
        if self.position >= len(self.order):
            self._reshuffle()
        idx = self.order[self.position]
        self.position += 1
        self.examples_seen += 1
        return self.examples[idx]

    def state_dict(self) -> Dict[str, Any]:
        return {
            "seed": self.seed,
            "rng_state": self.rng.getstate(),
            "order": list(self.order),
            "position": int(self.position),
            "epoch": int(self.epoch),
            "examples_seen": int(self.examples_seen),
            "dataset_length": len(self.examples),
        }

    def load_state_dict(self, state: Dict[str, Any]) -> None:
        if int(state.get("dataset_length", len(self.examples))) != len(self.examples):
            raise ValueError("Cannot restore train stream: checkpoint dataset length does not match current dataset length.")
        self.seed = int(state.get("seed", self.seed))
        self.rng.setstate(state["rng_state"])
        self.order = list(state["order"])
        self.position = int(state["position"])
        self.epoch = int(state["epoch"])
        self.examples_seen = int(state.get("examples_seen", 0))


train_stream = StatefulExampleStream(train_examples, CONFIG.seed)


def optimizer_update_and_lora_sync(step: int) -> Tuple[Path, Path, Dict[str, Optional[float]]]:
    grad_norms = {"solver": None, "failure": None}
    if CONFIG.max_grad_norm is not None and CONFIG.max_grad_norm > 0:
        solver_grad_norm = torch.nn.utils.clip_grad_norm_(adapter_named_parameters(CONFIG.solver_adapter_name).values(), CONFIG.max_grad_norm)
        failure_grad_norm = torch.nn.utils.clip_grad_norm_(adapter_named_parameters(CONFIG.failure_adapter_name).values(), CONFIG.max_grad_norm)
        grad_norms["solver"] = float(solver_grad_norm.detach().cpu())
        grad_norms["failure"] = float(failure_grad_norm.detach().cpu())
    solver_optimizer.step()
    failure_optimizer.step()
    solver_optimizer.zero_grad(set_to_none=True)
    failure_optimizer.zero_grad(set_to_none=True)
    solver_adapter_path, failure_adapter_path = save_reload_and_cleanup(f"{step:06d}")
    return solver_adapter_path, failure_adapter_path, grad_norms


def generate_eval_responses(example: Dict[str, Any], k: int) -> List[str]:
    return generate_with_vllm(
        messages=example["messages"],
        n=k,
        temperature=CONFIG.eval_temperature,
        top_p=CONFIG.eval_top_p,
        max_tokens=CONFIG.eval_max_completion_tokens,
    )


def generate_eval_batch(batch: Sequence[Dict[str, Any]], k: int) -> Dict[int, List[str]]:
    results: Dict[int, List[str]] = {}
    max_workers = max(1, min(len(batch), int(CONFIG.eval_batch_size)))
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {
            executor.submit(generate_eval_responses, example, k): idx
            for idx, example in enumerate(batch)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            results[idx] = future.result()
    return results


def evaluate_vllm(examples: Sequence[Dict[str, Any]], step: int, max_examples: Optional[int] = None) -> Dict[str, Any]:
    selected = list(examples[: max_examples or len(examples)])
    k = int(CONFIG.eval_generations)
    rows: List[Dict[str, Any]] = []
    prompt_rows: List[Dict[str, Any]] = []
    prompt_first_correct = 0
    prompt_any_correct = 0
    generation_correct = 0
    generation_total = 0
    parseable_total = 0

    for batch_start in range(0, len(selected), int(CONFIG.eval_batch_size)):
        batch = selected[batch_start: batch_start + int(CONFIG.eval_batch_size)]
        batch_outputs = generate_eval_batch(batch, k)
        for local_idx, example in enumerate(batch):
            responses = batch_outputs.get(local_idx, [])
            generation_results = []
            for generation_index, response in enumerate(responses):
                predicted = normalize_model_answer(response)
                is_correct = predicted is not None and predicted == example["ground_truth_normalized"]
                parseable_total += int(predicted is not None)
                generation_correct += int(is_correct)
                generation_total += 1
                row = {
                    "example_id": example.get("example_id"),
                    "url": example.get("url"),
                    "generation_index": generation_index,
                    "predicted_normalized": predicted,
                    "ground_truth_normalized": example.get("ground_truth_normalized"),
                    "correct": bool(is_correct),
                    "response": truncate_text(response, CONFIG.log_response_chars),
                }
                generation_results.append(row)
                rows.append(row)
            first_correct = bool(generation_results and generation_results[0].get("correct") is True)
            any_correct = any(bool(row.get("correct")) for row in generation_results)
            prompt_first_correct += int(first_correct)
            prompt_any_correct += int(any_correct)
            prompt_rows.append({
                "example_id": example.get("example_id"),
                "url": example.get("url"),
                "ground_truth_normalized": example.get("ground_truth_normalized"),
                "first_correct": first_correct,
                "any_correct": any_correct,
                "correct_count": int(sum(bool(row.get("correct")) for row in generation_results)),
                "generation_count": int(len(generation_results)),
            })

    prompt_total = len(selected)
    summary = {
        "step": int(step),
        "dataset": CONFIG.eval_dataset_name,
        "split": CONFIG.eval_split,
        "total_prompts": int(prompt_total),
        "total_generations": int(generation_total),
        "k": int(k),
        "eval_temperature": float(CONFIG.eval_temperature),
        "eval_top_p": float(CONFIG.eval_top_p),
        "pass_at_1": float(prompt_first_correct / prompt_total) if prompt_total else 0.0,
        "pass_at_k": float(prompt_any_correct / prompt_total) if prompt_total else 0.0,
        "avg_at_k": float(generation_correct / generation_total) if generation_total else 0.0,
        "parse_rate": float(parseable_total / generation_total) if generation_total else 0.0,
    }
    print("Evaluation summary:", json.dumps(summary, indent=2))
    write_jsonl({"tag": "eval", "algorithm": CONFIG.algorithm_name, "summary": summary, "prompt_rows": prompt_rows, "rows": rows})
    wandb_log_scalars(summary, prefix="eval", optimizer_step=step)
    if CONFIG.wandb_log_tables and rows:
        wandb_log_table("eval/generations", rows, optimizer_step=step)
        wandb_log_table("eval/prompts", prompt_rows, optimizer_step=step)
    return {"summary": summary, "prompt_rows": prompt_rows, "rows": rows}


def trainable_named_parameters() -> Dict[str, torch.nn.Parameter]:
    params = {}
    params.update(adapter_named_parameters(CONFIG.solver_adapter_name))
    params.update(adapter_named_parameters(CONFIG.failure_adapter_name))
    return params


def capture_adapter_state(adapter_name: str) -> Dict[str, torch.Tensor]:
    return {name: param.detach().cpu().clone() for name, param in adapter_named_parameters(adapter_name).items()}


def capture_trainable_state() -> Dict[str, Dict[str, torch.Tensor]]:
    return {
        "solver": capture_adapter_state(CONFIG.solver_adapter_name),
        "failure": capture_adapter_state(CONFIG.failure_adapter_name),
    }


def load_adapter_state(adapter_name: str, state: Dict[str, torch.Tensor]) -> None:
    params = adapter_named_parameters(adapter_name)
    with torch.no_grad():
        for name, tensor in state.items():
            if name not in params:
                raise KeyError(f"Checkpoint has {adapter_name} parameter not found in current model: {name}")
            params[name].copy_(tensor.to(device=params[name].device, dtype=params[name].dtype))


def load_trainable_state(state: Dict[str, Any]) -> None:
    if "solver" in state or "failure" in state:
        load_adapter_state(CONFIG.solver_adapter_name, state.get("solver", {}))
        load_adapter_state(CONFIG.failure_adapter_name, state.get("failure", {}))
    else:
        # Backward-compatible path for older single-adapter checkpoints.
        load_adapter_state(CONFIG.solver_adapter_name, state)


def capture_gradient_state() -> Dict[str, Dict[str, Optional[torch.Tensor]]]:
    return {
        "solver": {
            name: None if param.grad is None else param.grad.detach().cpu().clone()
            for name, param in adapter_named_parameters(CONFIG.solver_adapter_name).items()
        },
        "failure": {
            name: None if param.grad is None else param.grad.detach().cpu().clone()
            for name, param in adapter_named_parameters(CONFIG.failure_adapter_name).items()
        },
    }


def load_one_gradient_state(adapter_name: str, state: Dict[str, Optional[torch.Tensor]]) -> None:
    params = adapter_named_parameters(adapter_name)
    for name, grad in state.items():
        if name not in params:
            continue
        if grad is None:
            params[name].grad = None
        else:
            params[name].grad = grad.to(device=params[name].device, dtype=params[name].dtype).clone()


def load_gradient_state(state: Dict[str, Any]) -> None:
    if "solver" in state or "failure" in state:
        load_one_gradient_state(CONFIG.solver_adapter_name, state.get("solver", {}))
        load_one_gradient_state(CONFIG.failure_adapter_name, state.get("failure", {}))
    else:
        load_one_gradient_state(CONFIG.solver_adapter_name, state)


def move_optimizer_state_to_device(optimizer_obj) -> None:
    for state in optimizer_obj.state.values():
        for key, value in list(state.items()):
            if torch.is_tensor(value):
                state[key] = value.to(TRAIN_DEVICE)


def capture_rng_state() -> Dict[str, Any]:
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng_state(state: Dict[str, Any]) -> None:
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


def optional_scheduler_state_dict() -> Optional[Dict[str, Any]]:
    scheduler = globals().get("scheduler")
    if scheduler is None:
        return None
    return scheduler.state_dict()


def load_optional_scheduler_state(state: Optional[Dict[str, Any]]) -> None:
    scheduler = globals().get("scheduler")
    if scheduler is not None and state is not None:
        scheduler.load_state_dict(state)


def json_safe_training_state(state: Dict[str, Any]) -> Dict[str, Any]:
    safe = dict(state)
    if safe.get("adapter_path") is not None:
        safe["adapter_path"] = str(safe["adapter_path"])
    if safe.get("checkpoint_path") is not None:
        safe["checkpoint_path"] = str(safe["checkpoint_path"])
    return safe


def current_wandb_run_id() -> Optional[str]:
    if WANDB_RUN is None:
        return None
    return getattr(WANDB_RUN, "id", None)


def checkpoint_dir_for(kind: str, optimizer_step: int, completed_accumulations: int = 0) -> Path:
    if kind == "accumulation":
        label = f"checkpoint_step_{int(optimizer_step):06d}_accum_{int(completed_accumulations):02d}"
    else:
        label = f"checkpoint_step_{int(optimizer_step):06d}_optimizer"
    return CHECKPOINT_ROOT / label


def checkpoint_failure_adapter_path(checkpoint_dir: Path) -> Path:
    return Path(checkpoint_dir) / "failure_adapter"


def checkpoint_metadata(
    checkpoint_dir: Path,
    kind: str,
    optimizer_step: int,
    completed_accumulations: int,
) -> Dict[str, Any]:
    return {
        "checkpoint_path": str(checkpoint_dir),
        "checkpoint_kind": str(kind),
        "optimizer_step": int(optimizer_step),
        "completed_accumulations": int(completed_accumulations),
        "examples_seen": int(train_stream.state_dict().get("examples_seen", 0)),
        "adapter_path": str(checkpoint_adapter_path(checkpoint_dir)),
        "failure_adapter_path": str(checkpoint_failure_adapter_path(checkpoint_dir)),
        "algorithm": CONFIG.algorithm_name,
        "wandb_run_id": current_wandb_run_id(),
        "created_at": time.time(),
    }


def write_latest_checkpoint_pointer(metadata: Dict[str, Any]) -> None:
    tmp = LATEST_CHECKPOINT_PATH.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp.replace(LATEST_CHECKPOINT_PATH)


def cleanup_old_checkpoints(keep_path: Optional[Path] = None) -> None:
    keep = max(1, int(CONFIG.keep_last_checkpoints))
    checkpoints = sorted(
        [p for p in CHECKPOINT_ROOT.glob("checkpoint_step_*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    protected = {Path(keep_path).resolve()} if keep_path is not None else set()
    for stale in checkpoints[keep:]:
        if stale.resolve() in protected:
            continue
        shutil.rmtree(stale, ignore_errors=True)


def save_training_checkpoint(
    kind: str,
    optimizer_step: int,
    completed_accumulations: int,
    pending_step_records: Sequence[Dict[str, Any]],
) -> Optional[Path]:
    if not bool(CONFIG.checkpoint_enabled):
        return None

    checkpoint_dir = checkpoint_dir_for(kind, optimizer_step, completed_accumulations)
    tmp_dir = checkpoint_dir.with_name(checkpoint_dir.name + ".tmp")
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    if checkpoint_dir.exists():
        shutil.rmtree(checkpoint_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    adapter_dir = checkpoint_adapter_path(tmp_dir)
    failure_adapter_dir = checkpoint_failure_adapter_path(tmp_dir)
    save_named_lora_adapter(CONFIG.solver_adapter_name, adapter_dir)
    save_named_lora_adapter(CONFIG.failure_adapter_name, failure_adapter_dir)

    state_payload = {
        "optimizer_step": int(optimizer_step),
        "completed_accumulations": int(completed_accumulations),
        "pending_step_records": list(pending_step_records),
        "train_stream_state": train_stream.state_dict(),
        "trainable_state": capture_trainable_state(),
        "gradient_state": capture_gradient_state(),
        "solver_optimizer_state": solver_optimizer.state_dict(),
        "failure_optimizer_state": failure_optimizer.state_dict(),
        "scheduler_state": optional_scheduler_state_dict(),
        "rng_state": capture_rng_state(),
        "training_state": json_safe_training_state(training_state),
        "config": asdict(CONFIG),
    }
    torch.save(state_payload, checkpoint_state_path(tmp_dir))

    metadata = checkpoint_metadata(tmp_dir, kind, optimizer_step, completed_accumulations)
    tmp_meta = checkpoint_metadata_path(tmp_dir).with_suffix(".json.tmp")
    tmp_meta.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp_meta.replace(checkpoint_metadata_path(tmp_dir))

    tmp_dir.rename(checkpoint_dir)
    metadata["checkpoint_path"] = str(checkpoint_dir)
    metadata["adapter_path"] = str(checkpoint_adapter_path(checkpoint_dir))
    metadata["failure_adapter_path"] = str(checkpoint_failure_adapter_path(checkpoint_dir))
    tmp_meta2 = checkpoint_metadata_path(checkpoint_dir).with_suffix(".json.tmp")
    tmp_meta2.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp_meta2.replace(checkpoint_metadata_path(checkpoint_dir))
    write_latest_checkpoint_pointer(metadata)
    cleanup_old_checkpoints(checkpoint_dir)
    print(f"Saved {kind} checkpoint: {checkpoint_dir}")
    return checkpoint_dir


def load_training_checkpoint(checkpoint_dir: Path) -> Dict[str, Any]:
    checkpoint_dir = Path(checkpoint_dir)
    state_path = checkpoint_state_path(checkpoint_dir)
    if not state_path.exists():
        raise FileNotFoundError(f"Missing checkpoint state: {state_path}")
    payload = torch.load(state_path, map_location="cpu")
    load_trainable_state(payload["trainable_state"])
    solver_optimizer.load_state_dict(payload.get("solver_optimizer_state", payload.get("optimizer_state", {})))
    failure_optimizer.load_state_dict(payload.get("failure_optimizer_state", {}))
    move_optimizer_state_to_device(solver_optimizer)
    move_optimizer_state_to_device(failure_optimizer)
    load_optional_scheduler_state(payload.get("scheduler_state"))
    load_gradient_state(payload.get("gradient_state", {}))
    restore_rng_state(payload["rng_state"])
    train_stream.load_state_dict(payload["train_stream_state"])

    restored_state = dict(payload.get("training_state", {}))
    restored_state["optimizer_step"] = int(payload.get("optimizer_step", restored_state.get("optimizer_step", 0)))
    restored_state["completed_accumulations"] = int(payload.get("completed_accumulations", 0))
    restored_state["pending_step_records"] = list(payload.get("pending_step_records", []))
    restored_state["adapter_path"] = str(checkpoint_adapter_path(checkpoint_dir))
    restored_state["failure_adapter_path"] = str(checkpoint_failure_adapter_path(checkpoint_dir))
    restored_state["checkpoint_path"] = str(checkpoint_dir)
    restored_state["resumed_from_checkpoint"] = True
    restored_state["resume_optimizer_step"] = int(payload.get("optimizer_step", 0))
    restored_state["resume_completed_accumulations"] = int(payload.get("completed_accumulations", 0))

    global current_adapter_path, current_failure_adapter_path
    current_adapter_path = checkpoint_adapter_path(checkpoint_dir)
    current_failure_adapter_path = checkpoint_failure_adapter_path(checkpoint_dir)
    launch_vllm()
    reload_vllm_lora(current_adapter_path)
    print(
        "Restored checkpoint:", checkpoint_dir,
        "optimizer_step=", restored_state["optimizer_step"],
        "completed_accumulations=", restored_state["completed_accumulations"],
    )
    return restored_state


def run_stateful_stream_checkpoint_unit_tests() -> None:
    stream = StatefulExampleStream([{"x": i} for i in range(5)], seed=123)
    first = [next(stream)["x"] for _ in range(8)]
    state = stream.state_dict()
    future = [next(stream)["x"] for _ in range(8)]
    stream.load_state_dict(state)
    replay = [next(stream)["x"] for _ in range(8)]
    assert replay == future
    print("Stateful stream checkpoint tests passed.")


def run_rng_checkpoint_unit_tests() -> None:
    state = capture_rng_state()
    py_vals = [random.random() for _ in range(3)]
    np_vals = np.random.random(3)
    torch_vals = torch.rand(3)
    restore_rng_state(state)
    assert [random.random() for _ in range(3)] == py_vals
    assert np.allclose(np.random.random(3), np_vals)
    assert torch.allclose(torch.rand(3), torch_vals)
    restore_rng_state(state)
    print("RNG checkpoint tests passed.")


def run_trainable_state_roundtrip_unit_tests() -> None:
    state = capture_trainable_state()
    params = trainable_named_parameters()
    with torch.no_grad():
        for param in params.values():
            param.add_(0.001)
    load_trainable_state(state)
    for adapter_key, adapter_state in state.items():
        adapter_name = CONFIG.solver_adapter_name if adapter_key == "solver" else CONFIG.failure_adapter_name
        params_for_adapter = adapter_named_parameters(adapter_name)
        for name, tensor in adapter_state.items():
            assert torch.allclose(params_for_adapter[name].detach().cpu(), tensor, atol=0.0, rtol=0.0)
    print("Trainable state checkpoint tests passed.")


def run_checkpoint_metadata_unit_tests() -> None:
    meta = checkpoint_metadata(CHECKPOINT_ROOT / "checkpoint_step_000001_optimizer", "optimizer_step", 1, 0)
    required = {"checkpoint_path", "checkpoint_kind", "optimizer_step", "completed_accumulations", "examples_seen", "adapter_path", "failure_adapter_path", "algorithm"}
    assert required.issubset(meta.keys())
    print("Checkpoint metadata tests passed.")


run_stateful_stream_checkpoint_unit_tests()
run_rng_checkpoint_unit_tests()
run_trainable_state_roundtrip_unit_tests()
run_checkpoint_metadata_unit_tests()


def ensure_current_adapter_path() -> Path:
    global current_adapter_path
    existing = globals().get("current_adapter_path")
    if existing is not None:
        existing_path = Path(existing)
        if existing_path.exists():
            current_adapter_path = existing_path
            return current_adapter_path
        print(f"Tracked adapter path does not exist anymore: {existing_path}.")

    if RESUME_CHECKPOINT_PATH is not None and checkpoint_adapter_path(RESUME_CHECKPOINT_PATH).exists():
        current_adapter_path = checkpoint_adapter_path(RESUME_CHECKPOINT_PATH)
        launch_vllm()
        reload_vllm_lora(current_adapter_path)
        return current_adapter_path

    print("current_adapter_path is not defined. Launching vLLM and loading an initial LoRA adapter now.")
    launch_vllm()
    current_adapter_path, current_failure_adapter_path = save_reload_and_cleanup("initial")
    globals()["current_failure_adapter_path"] = current_failure_adapter_path
    return current_adapter_path


training_state = {
    "optimizer_step": 0,
    "adapter_path": str(ensure_current_adapter_path()),
    "failure_adapter_path": str(globals().get("current_failure_adapter_path", "")),
    "in_progress_step": None,
    "completed_accumulations": 0,
    "pending_step_records": [],
    "checkpoint_path": None,
    "resumed_from_checkpoint": False,
    "resume_optimizer_step": 0,
    "resume_completed_accumulations": 0,
}

if RESUME_CHECKPOINT_PATH is not None:
    restored_training_state = load_training_checkpoint(RESUME_CHECKPOINT_PATH)
    training_state.update(restored_training_state)
else:
    print("No resume checkpoint selected; starting from optimizer_step=0.")


def run_training_steps(num_optimizer_steps: int, tag: str = "train", run_validation: bool = True) -> None:
    if num_optimizer_steps <= 0:
        print("No optimizer steps requested.")
        return

    completed_target = int(training_state.get("optimizer_step", 0)) + int(num_optimizer_steps)
    while int(training_state.get("optimizer_step", 0)) < completed_target:
        if training_state.get("in_progress_step") is not None:
            next_step = int(training_state["in_progress_step"])
            completed_accumulations = int(training_state.get("completed_accumulations", 0))
            step_records = list(training_state.get("pending_step_records", []))
            print(f"\n=== Resuming optimizer step {next_step} ({tag}) after {completed_accumulations} accumulation(s) ===")
        else:
            next_step = int(training_state.get("optimizer_step", 0)) + 1
            completed_accumulations = 0
            step_records = []
            training_state["in_progress_step"] = next_step
            training_state["completed_accumulations"] = 0
            training_state["pending_step_records"] = []
            print(f"\n=== Starting optimizer step {next_step} ({tag}) ===")

        start_accum = completed_accumulations + 1
        for accum_idx in range(start_accum, CONFIG.gradient_accumulation_steps + 1):
            examples = [next(train_stream) for _ in range(int(CONFIG.train_prompt_batch_size))]
            prompt_records = train_on_prompt_batch(
                examples=examples,
                optimizer_step=next_step,
                accumulation_index=accum_idx,
                adapter_path=Path(training_state["adapter_path"]),
                tag=tag,
            )
            step_records.extend(prompt_records)
            training_state["completed_accumulations"] = accum_idx
            training_state["pending_step_records"] = step_records
            if CONFIG.checkpoint_every_accumulation:
                ckpt_path = save_training_checkpoint(
                    kind="accumulation",
                    optimizer_step=next_step,
                    completed_accumulations=accum_idx,
                    pending_step_records=step_records,
                )
                if ckpt_path is not None:
                    training_state["checkpoint_path"] = str(ckpt_path)

        step_summary = summarize_step_records(step_records)
        solver_adapter_path, failure_adapter_path, grad_norms = optimizer_update_and_lora_sync(next_step)
        training_state["adapter_path"] = str(solver_adapter_path)
        training_state["failure_adapter_path"] = str(failure_adapter_path)
        training_state["optimizer_step"] = next_step
        training_state["in_progress_step"] = None
        training_state["completed_accumulations"] = 0
        training_state["pending_step_records"] = []

        ckpt_path = None
        if CONFIG.checkpoint_every_optimizer_steps and next_step % int(CONFIG.checkpoint_every_optimizer_steps) == 0:
            ckpt_path = save_training_checkpoint(
                kind="optimizer_step",
                optimizer_step=next_step,
                completed_accumulations=0,
                pending_step_records=[],
            )
            if ckpt_path is not None:
                training_state["checkpoint_path"] = str(ckpt_path)

        if step_summary:
            step_summary["solver_grad_norm"] = grad_norms.get("solver")
            step_summary["failure_grad_norm"] = grad_norms.get("failure")
            step_summary["adapter_path"] = str(training_state["adapter_path"])
            step_summary["failure_adapter_path"] = str(training_state["failure_adapter_path"])
            step_summary["checkpoint_path"] = str(ckpt_path or training_state.get("checkpoint_path"))
            step_summary["checkpoint_kind"] = "optimizer_step" if ckpt_path is not None else None
            step_summary["checkpoint_saved"] = ckpt_path is not None
            step_summary["resumed_from_checkpoint"] = bool(training_state.get("resumed_from_checkpoint", False))
            step_summary["resume_optimizer_step"] = int(training_state.get("resume_optimizer_step", 0))
            step_summary["resume_completed_accumulations"] = int(training_state.get("resume_completed_accumulations", 0))
            step_summary["examples_seen"] = int(train_stream.state_dict().get("examples_seen", 0))
            wandb_summary_record = {"tag": f"{tag}_optimizer_summary", **step_summary}
            write_jsonl(wandb_summary_record)
            wandb_log_scalars(wandb_summary_record, prefix="train/optimizer", optimizer_step=next_step)
            if CONFIG.print_optimizer_summary_every_step:
                print_optimizer_step_summary(step_summary, grad_norms, Path(training_state["adapter_path"]))

        if run_validation and next_step % CONFIG.eval_every_steps == 0:
            evaluate_vllm(eval_examples, step=next_step, max_examples=CONFIG.eval_max_examples)
        if CONFIG.nvidia_smi_every_steps and next_step % CONFIG.nvidia_smi_every_steps == 0:
            print_nvidia_smi()

        gc.collect()
        torch.cuda.empty_cache()


## 10. One optimizer-step smoke test

This performs one real DAPO-Math-17K FEPO/Dr.GRPO optimizer step, reloads the solver LoRA adapter into vLLM, saves the failure LoRA separately, writes per-response JSONL logs, and can run a tiny AMC23 evaluation smoke test.


In [ ]:
if CONFIG.run_smoke_test and RESUME_CHECKPOINT_PATH is not None:
    print("Smoke training step skipped because a resume checkpoint is selected.")
    assert training_state.get("resumed_from_checkpoint") is True
    assert int(training_state.get("optimizer_step", 0)) >= int(RESUME_CHECKPOINT_METADATA.get("optimizer_step", 0))
    assert checkpoint_state_path(RESUME_CHECKPOINT_PATH).exists()
    assert checkpoint_adapter_path(RESUME_CHECKPOINT_PATH).exists()
elif CONFIG.run_smoke_test:
    LOG_PATH.unlink(missing_ok=True)
    run_training_steps(1, tag="smoke", run_validation=False)
    assert LOG_PATH.exists(), "Expected JSONL log file to be written."
    all_log_records = [json.loads(line) for line in LOG_PATH.read_text(encoding="utf-8").splitlines()]
    expected_response_records = CONFIG.num_generations * CONFIG.train_prompt_batch_size * CONFIG.gradient_accumulation_steps
    response_records = [r for r in all_log_records if r.get("algorithm") == CONFIG.algorithm_name and "generation_index" in r]
    first_records = response_records[:CONFIG.num_generations]
    required_keys = {
        "algorithm",
        "dataset",
        "problem",
        "example_id",
        "response",
        "generation_index",
        "prediction_normalized",
        "ground_truth_normalized",
        "has_parseable_answer",
        "reward",
        "anti_reward",
        "is_correct",
        "grpo_advantage",
        "failure_advantage",
        "group_reward_mean",
        "group_reward_std",
        "group_anti_reward_mean",
        "group_anti_reward_std",
        "active_response_count",
        "train_prompt_batch_index",
        "train_prompt_batch_size",
        "valid_token_count",
        "ignored_token_count",
        "solver_pg_loss",
        "failure_escape_kl_failed",
        "reference_kl_failed",
        "failed_token_count",
        "escape_alpha",
        "reference_beta",
        "solver_total_loss",
        "solver_scaled_loss",
        "failure_sft_loss",
        "failure_total_loss",
        "failure_scaled_loss",
        "failure_sft_token_count",
        "failure_is_sft_active",
        "failure_active_response_count",
        "token_ratio_mean",
        "token_log_ratio_abs_mean",
        "response_length",
        "adapter_path",
        "batch_correct",
        "batch_total",
        "batch_accuracy",
        "batch_parse_rate",
    }
    assert response_records, "Expected per-response training records."
    assert len(response_records) == expected_response_records
    assert first_records, "Expected per-response training records."
    assert len(first_records) == CONFIG.num_generations
    for record in first_records:
        assert required_keys.issubset(record.keys()), sorted(required_keys.difference(record.keys()))
        assert record["algorithm"] == CONFIG.algorithm_name
        assert record["dataset"] == "dapo-math-17k"
        assert float(record["reward"]) in {0.0, 1.0}
        assert float(record["anti_reward"]) in {-1.0, 1.0}
        assert abs(float(record["anti_reward"]) - (1.0 - 2.0 * float(record["reward"]))) < 1e-6
        assert record["generation_index"] in range(CONFIG.num_generations)
    rewards = [float(r["reward"]) for r in first_records]
    advantages = [float(r["grpo_advantage"]) for r in first_records]
    failure_advantages = [float(r["failure_advantage"]) for r in first_records]
    assert all(abs(fa + 2.0 * adv) < 1e-5 for fa, adv in zip(failure_advantages, advantages))
    if max(rewards) == min(rewards):
        assert all(abs(a) < 1e-6 for a in advantages)
        assert all(abs(a) < 1e-6 for a in failure_advantages)
    summary_records = [r for r in all_log_records if r.get("tag") == "smoke_optimizer_summary"]
    assert summary_records, "Expected smoke optimizer summary record."
    summary_record = summary_records[-1]
    required_summary_keys = {
        "optimizer_step",
        "adapter_path",
        "failure_adapter_path",
        "checkpoint_path",
        "checkpoint_kind",
        "solver_grad_norm",
        "failure_grad_norm",
        "resumed_from_checkpoint",
        "resume_optimizer_step",
        "resume_completed_accumulations",
        "examples_seen",
        "failure_sft_loss_mean",
        "failure_active_response_count_mean",
    }
    assert required_summary_keys.issubset(summary_record.keys()), sorted(required_summary_keys.difference(summary_record.keys()))
    checkpoint_due = bool(CONFIG.checkpoint_enabled) and (
        bool(CONFIG.checkpoint_every_accumulation)
        or (
            int(CONFIG.checkpoint_every_optimizer_steps or 0) > 0
            and int(training_state.get("optimizer_step", 0)) % int(CONFIG.checkpoint_every_optimizer_steps) == 0
        )
    )
    if checkpoint_due:
        assert LATEST_CHECKPOINT_PATH.exists(), "Expected latest checkpoint pointer to be written."
        latest_payload = json.loads(LATEST_CHECKPOINT_PATH.read_text(encoding="utf-8"))
        checkpoint_path = Path(latest_payload["checkpoint_path"])
        assert checkpoint_path.exists()
        assert checkpoint_state_path(checkpoint_path).exists()
        assert checkpoint_adapter_path(checkpoint_path).exists()
        assert checkpoint_failure_adapter_path(checkpoint_path).exists()
    else:
        print("Smoke checkpoint assertion skipped; checkpoint is not due at this step.")
    print("Smoke test wrote", len(LOG_PATH.read_text(encoding="utf-8").splitlines()), "JSONL lines.")
else:
    print("Smoke test skipped by config.")

if CONFIG.run_eval_smoke_test:
    eval_smoke = evaluate_vllm(eval_examples[: min(2, len(eval_examples))], step=training_state["optimizer_step"], max_examples=2)
    eval_summary = eval_smoke["summary"]
    assert {"pass_at_1", "pass_at_k", "avg_at_k"}.issubset(eval_summary.keys())
    assert eval_summary["k"] == CONFIG.eval_generations
    print("Evaluation smoke summary:", eval_summary)
else:
    print("Evaluation smoke test skipped by config.")


## 11. Full cumulative training run

Default target is `CONFIG.max_optimizer_steps` total optimizer steps. If the smoke test ran, this cell runs the remaining steps.


In [ ]:
if CONFIG.run_full_training:
    remaining_steps = CONFIG.max_optimizer_steps - training_state["optimizer_step"]
    run_training_steps(remaining_steps, tag="train", run_validation=True)
else:
    print("Full training skipped by config.")


## 12. Final AMC23 evaluation and save

The final solver and failure LoRA adapters are saved separately under FEPO DAPO-Math-17K Dr.GRPO `sign_flipped` `/kaggle/working` output directories.


In [ ]:
# final_eval = None
# if CONFIG.run_final_eval:
#     final_eval = evaluate_vllm(eval_examples, step=training_state["optimizer_step"], max_examples=CONFIG.eval_max_examples)
# else:
#     print("Final evaluation skipped by config.")

# save_named_lora_adapter(CONFIG.solver_adapter_name, FINAL_ADAPTER_DIR)
# save_named_lora_adapter(CONFIG.failure_adapter_name, FINAL_FAILURE_ADAPTER_DIR)

# print("Final solver adapter saved to:", FINAL_ADAPTER_DIR)
# print("Final failure adapter saved to:", FINAL_FAILURE_ADAPTER_DIR)
# print("Training log JSONL:", LOG_PATH)
# print("vLLM server log:", VLLM_LOG_PATH)
# if final_eval is not None:
#     print("Final AMC23 evaluation summary:", final_eval["summary"])

# finish_wandb_run(final_eval)


## 13. Optional cleanup

Run `stop_vllm()` after saving outputs if you want to stop the background vLLM process from the notebook.


In [ ]:
# def stop_vllm() -> None:
#     global vllm_process
#     if vllm_process is not None and vllm_process.poll() is None:
#         vllm_process.terminate()
#         try:
#             vllm_process.wait(timeout=30)
#         except subprocess.TimeoutExpired:
#             vllm_process.kill()
#             vllm_process.wait(timeout=30)
#         print("Stopped vLLM server.")
#     else:
#         print("No running vLLM subprocess tracked by this notebook.")
